# NB10 - Logit Robustness Benchmark

This notebook tests whether the main static-method findings from the primary LightGBM pipeline survive under an alternative benchmark pipeline: L2-penalised pooled logistic regression with balanced class weights and its own isotonic calibration.

The benchmark uses the same loan-month panel and the same rolling 12-month forward outcome. Its row-wise estimation format is related to person-period logistic models used in discrete-time event-history analysis, but the thesis target is not a current-interval hazard: it records whether distress occurs anywhere in `(t, t+12]`.

The re-evaluated methods are SCP, Mondrian FICO×LTV, and APS at `α = 0.10`, together with score-drift and matched-budget operational diagnostics. ACI, DtACI, coarse Mondrian partitions, and the local pocket audit are not rerun here.

**Secondary diagnostics:**
- Probability calibration: raw versus isotonic-calibrated logit scores.
- Coefficients: diagnostic interpretation of the linear benchmark.
- Operational translation: whether NB09's matched-budget identity/deficit pattern survives the benchmark pipeline.

---

## Structured notebook roadmap

| § | Title | What is done |
|---|---|---|
| **0** | Setup & Configuration | Dependencies, paths, constants, split metadata, and LightGBM reference baselines |
| **0.5** | Upstream Baseline (NB07/NB09) | Load locked NB07/NB09 reference findings |
| **1** | Pooled Logistic Benchmark | Fit the benchmark and document discrimination plus coefficient structure |
| **2** | Probability Calibration | Diagnose raw calibration and fit/apply isotonic recalibration |
| **3** | CP Threshold Calibration | Compute SCP, Mondrian FICO×LTV, and APS thresholds |
| **4** | Coverage Evaluation | Evaluate the three static methods across test regimes |
| **5** | Primary Pipeline vs Logit | Compare discrimination, coverage, CovGap, abstention, and score drift |
| **6** | Calibration Contribution | Compare raw and isotonic-calibrated benchmark scores |
| **7** | Operational Translation | Re-run the NB09 matched-budget decision-layer logic |
| **8** | Locked Findings | Record benchmark conclusions |
| **App.** | References | Sources cited in this notebook |

---

## Role of this notebook in the thesis

NB10 is a bundled robustness benchmark. The scorer, benchmark sample, preprocessing, fitted probability map, and diagnostic sample differ from the primary pipeline, while the split boundaries, underlying feature definitions, target, `α`, and metric definitions are held fixed.

Persistence therefore shows that the reported static-method patterns survive this alternative pipeline. It does not identify model class as the causal source of differences between the two pipelines.

---
## Section 0 · Setup & Configuration

In [ ]:
%pip install -q polars pyarrow scikit-learn scipy
import sklearn, polars as pl, scipy, numpy as np

In [ ]:
from __future__ import annotations

import gc, hashlib, json, sys, time, warnings, zlib
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
from scipy import stats

from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

IS_COLAB = "google.colab" in sys.modules or Path("/content").exists()
print(f"IS_COLAB : {IS_COLAB}")
print(f"Python   : {sys.version.split()[0]}")

IS_COLAB : True
Python   : 3.12.13


In [ ]:
# ── Path layout ──────────────────────────────
import os

from google.colab import drive
drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive/master_thesis")

PANEL_DIR    = DRIVE_ROOT / "data"    / "panel"
RESULTS_DIR  = DRIVE_ROOT / "results"
CP_ART_DIR   = DRIVE_ROOT / "cp_artifacts"
MANIFEST_DIR = DRIVE_ROOT / "manifests"
NB06_OUT_DIR = RESULTS_DIR / "nb06"
NB07_OUT_DIR = RESULTS_DIR / "nb07"
NB09_OUT_DIR = RESULTS_DIR / "nb09"
NB10_OUT_DIR = RESULTS_DIR / "nb10"
NB10_OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"NB10 output dir : {NB10_OUT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
NB10 output dir : /content/drive/MyDrive/master_thesis/results/nb10


In [ ]:
# ── Global constants ────────────────────────────────────────
ALPHA           = 0.10            # nominal miscoverage level (1 - 0.90 target coverage)
NOMINAL_COV     = 1.0 - ALPHA     # 0.90
SAMPLE_FRAC     = 0.05            # 5 % hash-deterministic loan-level sample
N_HASH_BUCKETS  = 1000            # hash space
N_KEEP          = round(SAMPLE_FRAC * N_HASH_BUCKETS)   # = 50 buckets kept
PRAC_SIG_PP     = 2.0             # practical significance threshold (pp)
COVGAP_MIN_CELL_N = 30           # min FICO×LTV cell size for CovGap / WGC stability

TEST_SPLITS  = ["test_subprime", "test_normal", "test_covid", "test_rate_hike"]
ALL_SPLITS   = ["calibration"] + TEST_SPLITS
SPLIT_LABELS = {
    "calibration"   : "Calibration (2005–06)",
    "test_subprime" : "Subprime (2007–12)",
    "test_normal"   : "Normal (2013–Sep 2019)",
    "test_covid"    : "COVID (2020–21)",
    "test_rate_hike": "Rate Hike (2022–23)",
}
MECH_LABELS = {
    "test_subprime" : "Tail Shift",
    "test_normal"   : "Stable",
    "test_covid"    : "Structural Shift",
    "test_rate_hike": "Composition Shift",
}

METH_LABELS = {
    "scp": "SCP",
    "mondrian_fico_ltv": "Mondrian (FICO×LTV)",
    "aps": "APS"
}

def psi_band(p):
    """PSI monitoring band (<0.10 small, 0.10-0.25 moderate, >0.25 substantial)."""
    if pd.isna(p):  return "n/a"
    if p < 0.10:    return "small"
    if p <= 0.25:   return "moderate"
    return "substantial"

def hash_keep_mask(lsn_iterable) -> np.ndarray:
    """5% crc32 loan-level keep-mask: crc32(lsn) % N_HASH_BUCKETS < N_KEEP.
    Single source of truth for the sample rule shared with NB05a/NB08/NB09."""
    return np.fromiter(
        (zlib.crc32(str(s).encode()) % N_HASH_BUCKETS < N_KEEP for s in lsn_iterable),
        dtype=bool,
    )

# ── Load upstream manifests ──────────────────────────────────────
panel_mfst_path = MANIFEST_DIR / "panel_manifest.json"
assert panel_mfst_path.exists(), "panel_manifest.json not found"
with open(panel_mfst_path) as f:
    PANEL_MANIFEST = json.load(f)

cp_mfst_path = MANIFEST_DIR / "cp_manifest.json"
assert cp_mfst_path.exists(), "cp_manifest.json not found"
with open(cp_mfst_path) as f:
    CP_MANIFEST = json.load(f)

FEATURE_COLS  = PANEL_MANIFEST["feature_cols"]   # 29 features (identical to NB04)
SUBGROUP_COLS = PANEL_MANIFEST["subgroup_cols"]   # ['fico_tier', 'ltv_bucket', 'fico_ltv_group', 'census_division']
# Target column name is governed by the panel manifest contract.
TARGET_COL    = PANEL_MANIFEST["target_col"]
assert TARGET_COL == "y", (
    f"Panel manifest target_col is '{TARGET_COL}', but the column references in this "
    f"notebook assume 'y'. Update those references to use TARGET_COL before proceeding."
)

print(f"Active model features : {len(FEATURE_COLS)}")
print(f"  {FEATURE_COLS}")
print(f"Subgroup columns      : {SUBGROUP_COLS}")
print(f"Alpha (1 - coverage)  : {ALPHA}")

# ── Load NB04 discriminative metrics (AUROC/AUPRC/Brier per split) ─────────────
nb04_manifest_path = MANIFEST_DIR / "model_manifest.json"
assert nb04_manifest_path.exists(), f"NB04 manifest not found: {nb04_manifest_path}"
with open(nb04_manifest_path) as f:
    NB04_MANIFEST = json.load(f)
LGBM_DISC = (
    NB04_MANIFEST.get("per_split_metrics")
    or NB04_MANIFEST.get("evaluation", {}).get("per_split_metrics", {})
)

# ── Load NB06/07 coverage metrics (mc, cov_y1, covgap, empty per method/split) ─

# Legacy NB07 manifest path from the executed pipeline.
# Current NB07 writes results/nb07/nb07_manifest.json.
# Here the manifest is only a fallback locator for NB06 coverage_metrics.json.
nb07_manifest_path = MANIFEST_DIR / "analysis_manifest.json"
assert nb07_manifest_path.exists(), f"NB07 manifest not found: {nb07_manifest_path}"
with open(nb07_manifest_path) as f:
    NB07_MANIFEST = json.load(f)
_cov_metrics_path = NB06_OUT_DIR / "coverage_metrics.json"
if not _cov_metrics_path.exists():
    _cov_metrics_path = Path(NB07_MANIFEST["upstream_inputs"]["coverage_metrics_json"])
assert _cov_metrics_path.exists(), f"coverage_metrics.json not found: {_cov_metrics_path}"
with open(_cov_metrics_path) as f:
    _LGBM_COV = json.load(f)

# ── Load NB05a score manifest (PSI, Brier/AUROC pre/post-iso, PAVA breakpoints) ─
score_manifest_path = MANIFEST_DIR / "score_manifest.json"
assert score_manifest_path.exists(), f"score_manifest.json not found: {score_manifest_path}"
with open(score_manifest_path) as f:
    SCORE_MANIFEST = json.load(f)

# ── Helper to safely extract a per-split metric from NB06 output ──────────
def _lgbm_cov(method_key, metric_key, split_key=None):

    metric_aliases = {
        "mc": ["mc", "marginal_cov", "marginal_coverage", "coverage", "set_coverage"],
        "marginal_cov": ["marginal_cov", "mc", "marginal_coverage", "coverage", "set_coverage"],
        "cov_y0": ["cov_y0", "coverage_y0", "class0_coverage"],
        "cov_y1": ["cov_y1", "coverage_y1", "class1_coverage", "positive_class_coverage"],
        "empty_rate": ["empty_rate", "empty_set_rate", "empty"],
        "covgap_pp": ["covgap_pp", "coverage_gap_pp", "class_conditional_gap_pp"],
    }

    def _one(split):
        if metric_key in {"covgap_pp", "coverage_gap_pp", "class_conditional_gap_pp"}:
            try:
                return float(_LGBM_COV["covgap"][split][method_key]["covgap_pp"])
            except (KeyError, TypeError, ValueError):
                return float("nan")

        d = _LGBM_COV.get("global", {}).get(split, {}).get(method_key, {})
        if not isinstance(d, dict):
            return float("nan")

        for k in metric_aliases.get(metric_key, [metric_key]):
            if k in d and d[k] is not None:
                try:
                    return float(d[k])
                except (TypeError, ValueError):
                    return float("nan")

        return float("nan")

    if split_key is not None:
        return _one(split_key)

    return {s: _one(s) for s in TEST_SPLITS}

# ── Mondrian FICO×LTV threshold spread from cp_manifest calibration_summary ───
_mond_spread = CP_MANIFEST.get("calibration_summary", {}).get(
    "mondrian_fico_ltv_threshold_ratio_observed", None)
if _mond_spread is None:
    # Compute from stored thresholds as fallback
    _mond_dict = CP_MANIFEST.get("mondrian", {}).get("fico_ltv", {})
    _mond_q = [float(v) for k, v in _mond_dict.items()
               if k != "Sentinel"]
    _mond_spread = max(_mond_q) / min(_mond_q) if _mond_q and min(_mond_q) > 0 else float("nan")
LGBM_MONDRIAN_SPREAD = float(_mond_spread)

def _require_any(*sources_and_keys):
    """
    Return the first available value from one or more dictionaries.

    Usage:
        _require_any((dict_a, "key1", "key2"), (dict_b, "key3"))
    """
    tried = []
    for item in sources_and_keys:
        d, *keys = item
        if not isinstance(d, dict):
            continue
        for k in keys:
            tried.append(k)
            if k in d and d[k] is not None:
                return d[k]
    raise KeyError(f"None of {tried} found in the supplied manifest blocks.")


def _optional_any(default, *sources_and_keys):
    try:
        return _require_any(*sources_and_keys)
    except KeyError:
        return default


# ── Calibration-split class-conditional coverage from cp_manifest ──────────
_cal_sum = CP_MANIFEST.get("calibration_summary", {})
LGBM_SCP_COV_Y1_CAL = float(_cal_sum["scp_coverage_y1_cp_cal"])
LGBM_APS_COV_Y1_CAL = float(_cal_sum["aps_coverage_y1_cp_cal"])

# ── LightGBM SCP/APS thresholds from cp_manifest ─────────────────────────
LGBM_Q_SCP = float(CP_MANIFEST["scp"]["q_hat"])
LGBM_Q_APS = float(CP_MANIFEST["aps"]["q_hat"])

# ── Brier/AUROC diagnostics from SCORE_MANIFEST / NB05a ───────────────────
_score_diag = SCORE_MANIFEST.get("calibration_diagnostics", {})

LGBM_BRIER_PRE = float(_require_any(
    (_score_diag, "brier_pre_oos", "brier_pre"),
    (SCORE_MANIFEST, "brier_pre_oos", "brier_pre"),
))

LGBM_BRIER_POST = float(_require_any(
    (_score_diag, "brier_post"),
    (SCORE_MANIFEST, "brier_post"),
))

LGBM_BRIER_IMPROVE_PCT = (
    (LGBM_BRIER_PRE - LGBM_BRIER_POST) / LGBM_BRIER_PRE * 100
    if LGBM_BRIER_PRE != 0
    else float("nan")
)

LGBM_AUROC_DELTA = float(_require_any(
    (_score_diag, "auroc_delta"),
    (SCORE_MANIFEST, "auroc_delta"),
))

# PAVA breakpoint count is an optional descriptive diagnostic.
LGBM_PAVA_BP = _optional_any(
    None,
    (SCORE_MANIFEST, "pava_breakpoints", "n_pava_breakpoints"),
    (_score_diag, "pava_breakpoints", "n_pava_breakpoints"),
)
LGBM_PAVA_BP = None if LGBM_PAVA_BP is None else int(LGBM_PAVA_BP)

# ── PSI from SCORE_MANIFEST (try multiple plausible key names) ────────────
_psi_block = (SCORE_MANIFEST.get("psi_score_scp")
              or SCORE_MANIFEST.get("psi_scp")
              or SCORE_MANIFEST.get("psi", {}))
LGBM_PSI = {s: float(_psi_block.get(s, float("nan"))) for s in TEST_SPLITS}

# ── Assemble LGBM_REF from dynamic sources ──────────────────────────────
# Discriminative metrics from NB04 model_manifest per_split_metrics.
# Coverage metrics from NB06 coverage_metrics.json (via NB07 manifest).
# PSI from NB05a score_manifest.json.
LGBM_REF = {
    "auroc": {
        s: LGBM_DISC.get(s, {}).get("auroc", float("nan"))
        for s in ALL_SPLITS
    },
    "auprc": {
        s: LGBM_DISC.get(s, {}).get("ap", float("nan"))
        for s in ALL_SPLITS
    },
    "brier": {
        s: LGBM_DISC.get(s, {}).get("brier", float("nan"))
        for s in ALL_SPLITS
    },

    "mc_scp": {
        s: _lgbm_cov("scp", "mc")[s] * 100
        for s in TEST_SPLITS
    },
    "mc_mond": {
        s: _lgbm_cov("mondrian_fico_ltv", "mc")[s] * 100
        for s in TEST_SPLITS
    },
    "mc_aps": {
        s: _lgbm_cov("aps", "mc")[s] * 100
        for s in TEST_SPLITS
    },

    "cov_y1_scp": {
        s: _lgbm_cov("scp", "cov_y1")[s] * 100
        for s in TEST_SPLITS
    },
    "cov_y1_mond": {
        s: _lgbm_cov("mondrian_fico_ltv", "cov_y1")[s] * 100
        for s in TEST_SPLITS
    },
    "cov_y1_aps": {
        s: _lgbm_cov("aps", "cov_y1")[s] * 100
        for s in TEST_SPLITS
    },

    "covgap_scp": {
        s: _lgbm_cov("scp", "covgap_pp")[s]
        for s in TEST_SPLITS
    },
    "covgap_mond": {
        s: _lgbm_cov("mondrian_fico_ltv", "covgap_pp")[s]
        for s in TEST_SPLITS
    },
    "covgap_aps": {
        s: _lgbm_cov("aps", "covgap_pp")[s]
        for s in TEST_SPLITS
    },

    "empty_scp": {
        s: _lgbm_cov("scp", "empty_rate")[s] * 100
        for s in TEST_SPLITS
    },
    "empty_mond": {
        s: _lgbm_cov("mondrian_fico_ltv", "empty_rate")[s] * 100
        for s in TEST_SPLITS
    },
    "empty_aps": {
        s: _lgbm_cov("aps", "empty_rate")[s] * 100
        for s in TEST_SPLITS
    },

    "psi_scp": LGBM_PSI,
}

# ── Mean score shift from SCORE_MANIFEST / CP_MANIFEST ───────────────────
_mean_score_block = (
    SCORE_MANIFEST.get("mean_score_scp")
    or CP_MANIFEST.get("mean_score_scp")
    or {}
)

LGBM_MEAN_SCORE_SCP = {
    s: float(_mean_score_block.get(s, float("nan")))
    for s in ALL_SPLITS
}

_LGBM_MEAN_SCORE_CAL = LGBM_MEAN_SCORE_SCP.get("calibration", float("nan"))

LGBM_MEAN_SCORE_SHIFT_PCT = {
    s: (
        (LGBM_MEAN_SCORE_SCP[s] / _LGBM_MEAN_SCORE_CAL - 1.0) * 100.0
        if pd.notna(LGBM_MEAN_SCORE_SCP.get(s, float("nan")))
        and pd.notna(_LGBM_MEAN_SCORE_CAL)
        and _LGBM_MEAN_SCORE_CAL != 0
        else float("nan")
    )
    for s in TEST_SPLITS
}

# ── NB09 equal-review-rate Δ reference ────────────────────────────────────
_er_path = NB09_OUT_DIR / "nb09_equal_rate_comparison.csv"
assert _er_path.exists(), f"NB09 equal-rate comparison CSV not found: {_er_path}"
_er_df = pd.read_csv(_er_path)

def _nb09_delta(method_key, split_key):
    vals = _er_df.loc[
        (_er_df["split"] == split_key) &
        (_er_df["method"] == method_key),
        "delta_pp"
    ]
    return float(vals.iloc[0]) if len(vals) else float("nan")


NB09_DELTA_REF = {
    "scp": {s: 0.000 for s in TEST_SPLITS},
    "aps": {s: 0.000 for s in TEST_SPLITS},
    "mondrian_fico_ltv": {
        s: _nb09_delta("mondrian_fico_ltv", s)
        for s in TEST_SPLITS
    },
}

print("\n✓  All manifests loaded.")
print(f"  LGBM_Q_SCP={LGBM_Q_SCP:.8f}  LGBM_Q_APS={LGBM_Q_APS:.8f}")
print(f"  LGBM Mondrian FICO×LTV spread: {LGBM_MONDRIAN_SPREAD:.2f}×")
print(f"  LGBM Brier pre→post (OOS): {LGBM_BRIER_PRE:.6f} → {LGBM_BRIER_POST:.6f}  "
      f"({LGBM_BRIER_IMPROVE_PCT:.1f}% improvement)")
if LGBM_PAVA_BP is None:
    print("  LGBM PAVA breakpoints: unavailable")
else:
    print(f"  LGBM PAVA breakpoints: {LGBM_PAVA_BP}")
print(f"  LGBM PSI (score_scp): {LGBM_PSI}")

Active model features : 29
  ['cltv_missing', 'dti_missing', 'dti_missing_non_relief_refi', 'fico_missing', 'is_30dpd_at_t', 'is_multi_borrower', 'ltv_missing', 'first_time_homebuyer_flag', 'occupancy_status', 'number_of_units', 'loan_purpose', 'vintage_quarter', 'property_type', 'vintage_year', 'census_division', 'months_since_last_dlq', 'n_times_30dpd_last_12m', 'max_dlq_last_12m', 'original_dti', 'original_ltv', 'rate_spread', 'original_cltv', 'original_loan_term', 'credit_score', 'original_upb', 'original_interest_rate', 'current_interest_rate', 'current_upb', 'upb_rel_change_3m']
Subgroup columns      : ['fico_tier', 'ltv_bucket', 'fico_ltv_group', 'census_division']
Alpha (1 - coverage)  : 0.1

✓  All manifests loaded.
  LGBM_Q_SCP=0.01622366  LGBM_Q_APS=0.89995086
  LGBM Mondrian FICO×LTV spread: 128.71×
  LGBM Brier pre→post (OOS): 0.064263 → 0.008616  (86.6% improvement)
  LGBM PAVA breakpoints: 516
  LGBM PSI (score_scp): {'test_subprime': 0.006231, 'test_normal': 0.08912, 't

In [ ]:
# ── §0.5 · Upstream baseline ────────────────────────────────────

print("=" * 100)
print("§0.5  UPSTREAM BASELINE (NB07/NB09) - benchmarked against logit in §4-§7")
print("=" * 100)

print("\n  NB09 Operational Δ reference (NB09 Finding D2):")
for method, label in [("scp", "SCP"), ("mondrian_fico_ltv", "Mondrian"), ("aps", "APS")]:
    parts = "  ".join(f"{SPLIT_LABELS[s][:12]}: {v:+.3f}pp"
                      for s, v in NB09_DELTA_REF[method].items())
    print(f"    {label:<12}: {parts}")

§0.5  UPSTREAM BASELINE (NB07/NB09) - benchmarked against logit in §4-§7

  NB09 Operational Δ reference (NB09 Finding D2):
    SCP         : Subprime (20: +0.000pp  Normal (2013: +0.000pp  COVID (2020–: +0.000pp  Rate Hike (2: +0.000pp
    Mondrian    : Subprime (20: -8.739pp  Normal (2013: -5.818pp  COVID (2020–: -3.732pp  Rate Hike (2: -5.426pp
    APS         : Subprime (20: +0.000pp  Normal (2013: +0.000pp  COVID (2020–: +0.000pp  Rate Hike (2: +0.000pp


---
## Section 1 · Pooled Logistic Benchmark

### Methodological framing

The benchmark is a pooled logistic regression on loan-month observations. Its row-wise estimation format is related to person-period logistic models used in discrete-time event-history analysis, but the thesis target is a rolling 12-month forward event indicator.

The primary LightGBM model is also fitted row-wise. This keeps the analysis unit and target aligned, but NB10 is not a controlled model-class substitution: scorer, training sample, preprocessing, probability map, and diagnostic sample differ. Differences should therefore be interpreted as benchmark-pipeline differences, not effects attributable solely to linear versus nonlinear model class.

**Feature engineering for logit:**
- **String categoricals** (`first_time_homebuyer_flag`, `occupancy_status`, `loan_purpose`, `property_type`, `census_division`) → `OneHotEncoder(drop='first', handle_unknown='ignore')`.
- **Integer categoricals** (`vintage_year`, `vintage_quarter`, `number_of_units`) → treated as numeric after standard scaling, unlike the primary LightGBM pipeline.
- **Numeric features** → median imputation followed by standard scaling; existing missingness indicators retain explicit missingness information.
- **`upb_rel_change_3m`** → populated upstream only for non-modified observations with `loan_age >= 10` and a positive three-month-prior UPB. The age guard ensures `t-3` also clears the six-month origination UPB-masking window. NB10 median-imputes missing values and adds `upb_rel_change_3m_missing`.

**Class imbalance:** `class_weight='balanced'` is used. The positive-to-negative weight ratio** approximately `N_neg/N_pos ≈ 81.4`; the absolute balanced weights on the fitted sample are about 41.2 for positives and 0.506 for negatives. The stored §1.1 line “Positive-class weight ≈ 81.4” should therefore be read as the relative weight ratio. Reweighting changes the fitted probability scale, so §2 evaluates calibration empirically before the main conformal comparison.

In [ ]:
# ── §1.1 · Load 5 % hash-deterministic training sample ────────────────────────

LOAD_COLS = FEATURE_COLS + ["y", "loan_sequence_number", "obs_year"]

train_dir = PANEL_DIR / "split=train"
assert train_dir.exists(), f"Training split not found: {train_dir}"
train_files = sorted(train_dir.glob("*.parquet"))
print(f"Training Parquet files : {len(train_files)}")

# crc32 loan-level hash sample
t0 = time.time()
df_raw = (
    pl.scan_parquet([str(f) for f in train_files])
    .filter(
        pl.col("loan_sequence_number")
        .cast(pl.Utf8)
        .map_elements(
            lambda s: zlib.crc32(s.encode()) % N_HASH_BUCKETS < N_KEEP,
            return_dtype=pl.Boolean,
        )
    )
    .select(LOAD_COLS)
    .collect()
)
print(f"5 % sample loaded in {time.time()-t0:.1f}s  |  {len(df_raw):,} rows")

# ── Split into fit (obs_year < 2004) and val (obs_year == 2004) ───────────────
df_fit = df_raw.filter(pl.col("obs_year") < 2004)
df_val = df_raw.filter(pl.col("obs_year") == 2004)

N_FIT = len(df_fit); N_FIT_POS = int(df_fit["y"].sum())
N_VAL = len(df_val); N_VAL_POS = int(df_val["y"].sum())

print(f"\nFit rows  (obs_year < 2004) : {N_FIT:>10,}  pos={N_FIT_POS:,}  ({N_FIT_POS/N_FIT*100:.4f}%)")
print(f"Val rows  (obs_year == 2004) : {N_VAL:>10,}  pos={N_VAL_POS:,}  ({N_VAL_POS/N_VAL*100:.4f}%)")
print(f"Positive-class weight ≈ {(N_FIT - N_FIT_POS) / N_FIT_POS:.1f}  (class_weight='balanced' will target this)")
del df_raw; gc.collect()

Training Parquet files : 1
5 % sample loaded in 56.7s  |  12,385,067 rows

Fit rows  (obs_year < 2004) :  8,303,131  pos=100,759  (1.2135%)
Val rows  (obs_year == 2004) :  4,081,936  pos=40,924  (1.0026%)
Positive-class weight ≈ 81.4  (class_weight='balanced' will target this)


0

In [ ]:
# ── §1.2 · Feature preprocessing pipeline ─────────────────────────────────────

STRING_CATS = ["first_time_homebuyer_flag", "occupancy_status", "loan_purpose",
               "property_type", "census_division"]
# Remaining features (binary flags, numeric, int categoricals) treated as numeric
NUMERIC_FEATS = [f for f in FEATURE_COLS if f not in STRING_CATS]
# upb_rel_change_3m_missing: logit-specific indicator (not in LightGBM feature set)
UPB_MISSING_COL = "upb_rel_change_3m_missing"

def prepare_X(df: pl.DataFrame) -> np.ndarray:
    # Builds raw (n, p+1) feature matrix from a Polars DataFrame.
    # Adds upb_rel_change_3m_missing binary indicator before returning.
    # No scaling/encoding - those are handled inside the sklearn Pipeline.
    # Add missingness indicator for upb_rel_change_3m
    df = df.with_columns(
        pl.col("upb_rel_change_3m").is_null().cast(pl.Int8).alias(UPB_MISSING_COL)
    )
    # upb_rel_change_3m NULLs flow through to the numeric pipeline's
    # SimpleImputer(strategy="median"); the indicator above preserves the
    # missingness signal alongside the median-imputed value.
    ALL_COLS = NUMERIC_FEATS + [UPB_MISSING_COL]
    return df.select(ALL_COLS + STRING_CATS).to_numpy(allow_copy=True)

def get_feature_names(n_string_cats: list[str], ohe_categories: list) -> list[str]:
    # Return feature names in the order the ColumnTransformer outputs them.
    # 1. Numeric (incl. upb_rel_change_3m_missing)
    num_names = NUMERIC_FEATS + [UPB_MISSING_COL]
    # 2. OHE names (drop='first' -> n_categories - 1 per feature)
    ohe_names = []
    for feat, cats in zip(STRING_CATS, ohe_categories):
        for cat in cats[1:]:  # skip first (dropped)
            ohe_names.append(f"{feat}__{cat}")
    return num_names + ohe_names

# ── Build Pipeline ─────────────────────────────────────────────────────────────
numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale",  StandardScaler()),
])
ohe_pipe = Pipeline([
    ("ohe", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)),
])

ALL_INPUT_COLS = NUMERIC_FEATS + [UPB_MISSING_COL] + STRING_CATS
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, list(range(len(NUMERIC_FEATS) + 1))),                          # numeric cols by index
        ("cat", ohe_pipe,     list(range(len(NUMERIC_FEATS) + 1, len(ALL_INPUT_COLS)))),      # string cat cols by index
    ],
    remainder="drop",
)

# ── Fit preprocessor on FIT rows ─────────────────────────────────────────────
print("Fitting preprocessor on fit rows …")
t0 = time.time()
X_fit_raw = prepare_X(df_fit)
y_fit      = df_fit["y"].to_numpy().astype(np.int32)
preprocessor.fit(X_fit_raw)
X_fit = preprocessor.transform(X_fit_raw).astype(np.float32)
print(f"X_fit shape  : {X_fit.shape}  ({X_fit.nbytes / 1e9:.2f} GB)")

X_val_raw = prepare_X(df_val)
y_val     = df_val["y"].to_numpy().astype(np.int32)
X_val     = preprocessor.transform(X_val_raw).astype(np.float32)
print(f"X_val shape  : {X_val.shape}")

# Extract feature names for coefficient analysis
ohe_categories = preprocessor.named_transformers_["cat"]["ohe"].categories_
FEAT_NAMES_ALL = get_feature_names(STRING_CATS, ohe_categories)
print(f"Total features after encoding: {X_fit.shape[1]}  (named: {len(FEAT_NAMES_ALL)})")
del X_fit_raw, X_val_raw; gc.collect()

Fitting preprocessor on fit rows …
X_fit shape  : (8303131, 45)  (1.49 GB)
X_val shape  : (4081936, 45)
Total features after encoding: 45  (named: 45)


105

In [ ]:
# ── §1.3 · Regularisation (C) selection via validation grid search ─────────────
# A 3 % hash-subsample of the fit rows is used for rapid C selection. Each
# candidate C is fitted on this subsample and evaluated on the fixed obs_year == 2004
# validation split. This is a validation-set grid search, not k-fold cross-validation.
#
# Evaluation metric: validation AUROC on the obs_year == 2004 validation set.
# C range: {1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0} - six values spanning five decades.
C_GRID_SAMPLE_FRAC = 0.03   # 3 % subsample of fit rows for rapid C grid search
n_buckets_sub      = 1000
n_keep_sub         = int(C_GRID_SAMPLE_FRAC * n_buckets_sub)

sub_mask = (
    pl.Series("lsn", df_fit["loan_sequence_number"].to_list())
    .cast(pl.Utf8).hash(seed=99) % n_buckets_sub < n_keep_sub
).to_numpy()

X_sub = X_fit[sub_mask]
y_sub = y_fit[sub_mask]
print(f"3 % subsample: {len(X_sub):,} rows  (pos rate {y_sub.mean()*100:.3f}%)")

C_GRID  = [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0]
results = []
for C in C_GRID:
    t0 = time.time()
    lr = LogisticRegression(
        C=C, penalty="l2", solver="lbfgs", max_iter=500,
        class_weight="balanced", random_state=42, n_jobs=-1,
    )
    lr.fit(X_sub, y_sub)
    p_val = lr.predict_proba(X_val)[:, 1]
    auroc = roc_auc_score(y_val, p_val)
    results.append({"C": C, "val_auroc": auroc, "t": time.time()-t0})
    print(f"  C={C:<8}  val_AUROC = {auroc:.6f}  ({time.time()-t0:.1f}s)")

best = max(results, key=lambda r: r["val_auroc"])
BEST_C = best["C"]
print(f"\n✓  Best C = {BEST_C}  (val_AUROC = {best['val_auroc']:.6f})")
if BEST_C in (min(C_GRID), max(C_GRID)):
    _spread = max(r["val_auroc"] for r in results) - min(r["val_auroc"] for r in results)
del X_sub, y_sub; gc.collect()

3 % subsample: 249,996 rows  (pos rate 1.166%)
  C=0.0001    val_AUROC = 0.915598  (2.5s)
  C=0.001     val_AUROC = 0.913392  (2.5s)
  C=0.01      val_AUROC = 0.912020  (3.1s)
  C=0.1       val_AUROC = 0.911906  (3.8s)
  C=1.0       val_AUROC = 0.911996  (3.8s)
  C=10.0      val_AUROC = 0.911984  (3.8s)

✓  Best C = 0.0001  (val_AUROC = 0.915598)


162

In [ ]:
# ── §1.4 · Final logit training on full 5 % fit-training sample ───────────────
# solver = 'lbfgs'  : standard limited-memory quasi-Newton solver for
#                     large-scale smooth optimisation (Liu & Nocedal 1989).
# max_iter = 1000   : sufficient for lbfgs convergence at C = BEST_C.
# class_weight='balanced': see §1 markdown for the imbalance/calibration rationale
#                     (N_neg/N_pos ratio printed in §1.1).

print(f"Training LogisticRegression (C={BEST_C}, class_weight='balanced') …")
t0 = time.time()
LOGIT = LogisticRegression(
    C=BEST_C, penalty="l2", solver="lbfgs", max_iter=1000,
    class_weight="balanced", random_state=42, n_jobs=-1, tol=1e-4,
)
LOGIT.fit(X_fit, y_fit)
elapsed = time.time() - t0

p_val_final = LOGIT.predict_proba(X_val)[:, 1]
val_auroc   = roc_auc_score(y_val, p_val_final)
val_auprc   = average_precision_score(y_val, p_val_final)

print(f"\n✓  Training complete in {elapsed:.1f}s")
print(f"   Val AUROC (2004 holdout) : {val_auroc:.6f}")
print(f"   Val AUPRC (2004 holdout) : {val_auprc:.6f}")
print(f"   Iterations used          : {LOGIT.n_iter_[0]}")
print(f"   Intercept                : {LOGIT.intercept_[0]:.6f}")

del X_fit, X_val; gc.collect()

Training LogisticRegression (C=0.0001, class_weight='balanced') …

✓  Training complete in 28.6s
   Val AUROC (2004 holdout) : 0.921329
   Val AUPRC (2004 holdout) : 0.287956
   Iterations used          : 37
   Intercept                : -1.359995


27

In [ ]:
# ── §1.5 · Per-split performance evaluation ────────────────────────────────────
# Evaluated on the same 5 % loan-level hash sample as NB08/09 for comparability.
# p_hat_logit_raw = raw logit P(Y=1|X) before isotonic recalibration.

def load_split_sample(split: str, cols: list[str]) -> pl.DataFrame:
    split_dir = PANEL_DIR / f"split={split}"
    assert split_dir.exists(), f"Split not found: {split_dir}"
    return (
        pl.scan_parquet(str(split_dir / "*.parquet"))
        .filter(
            pl.col("loan_sequence_number")
            .cast(pl.Utf8)
            .map_elements(
                lambda s: zlib.crc32(s.encode()) % N_HASH_BUCKETS < N_KEEP,
                return_dtype=pl.Boolean,
            )
        )
        .select(cols)
        .collect()
    )

EVAL_COLS = list(dict.fromkeys(
    FEATURE_COLS + ["y", "loan_sequence_number", "obs_year"] + SUBGROUP_COLS
))

EVAL_RESULTS: dict = {}
_CHUNK = 2_000_000

for split in ALL_SPLITS:
    t0 = time.time()
    df_sp   = load_split_sample(split, EVAL_COLS)           # now only the 5% sample
    y_sp    = df_sp["y"].to_numpy().astype(np.int32)
    df_slim = df_sp.select(["loan_sequence_number", "obs_year", "fico_ltv_group"])

    _p_parts: list = []
    for _lo in range(0, len(df_sp), _CHUNK):
        _chunk = df_sp[_lo : _lo + _CHUNK]
        _X     = preprocessor.transform(prepare_X(_chunk))
        _p_parts.append(LOGIT.predict_proba(_X)[:, 1].astype(np.float32))
        del _chunk, _X
        gc.collect()
    del df_sp
    gc.collect()
    p_hat = np.concatenate(_p_parts)
    del _p_parts

    auroc  = roc_auc_score(y_sp, p_hat)
    auprc  = average_precision_score(y_sp, p_hat)
    brier  = float(brier_score_loss(y_sp, p_hat))

    EVAL_RESULTS[split] = {
        "p_hat_raw" : p_hat,
        "y"         : y_sp,
        "n_rows"    : len(y_sp),
        "pos_rate"  : float(y_sp.mean()),
        "auroc"     : auroc,
        "auprc"     : auprc,
        "brier"     : brier,
        "df"        : df_slim,
    }
    print(f"  {split:<18}  n={len(y_sp):>10,}  pos={y_sp.mean()*100:.4f}%  "
          f"AUROC={auroc:.4f}  AUPRC={auprc:.4f}  Brier={brier:.6f}  ({time.time()-t0:.0f}s)")

print()
# Comparison with LightGBM
print(f"{'Split':<20} {'Logit AUROC':>12}  {'LGBM AUROC':>12}  {'Delta':>8}")
print("-" * 58)
for spl in ALL_SPLITS:
    lgbm_a = LGBM_REF["auroc"].get(spl, float("nan"))
    logit_a = EVAL_RESULTS[spl]["auroc"]
    delta   = logit_a - lgbm_a
    print(f"  {spl:<20}  {logit_a:.4f}      {lgbm_a:.4f}      {delta:+.4f}")

  calibration         n= 8,972,803  pos=1.1031%  AUROC=0.9089  AUPRC=0.2865  Brier=0.066746  (68s)
  test_subprime       n=31,692,753  pos=2.5317%  AUROC=0.8810  AUPRC=0.2961  Brier=0.072213  (371s)
  test_normal         n=37,785,626  pos=1.5274%  AUROC=0.8724  AUPRC=0.2373  Brier=0.053625  (1185s)
  test_covid          n=10,079,140  pos=1.6672%  AUROC=0.8216  AUPRC=0.1500  Brier=0.050462  (337s)
  test_rate_hike      n=15,312,169  pos=1.0403%  AUROC=0.8638  AUPRC=0.1403  Brier=0.040534  (564s)

Split                 Logit AUROC    LGBM AUROC     Delta
----------------------------------------------------------
  calibration           0.9089      0.9140      -0.0052
  test_subprime         0.8810      0.8833      -0.0023
  test_normal           0.8724      0.8723      +0.0000
  test_covid            0.8216      0.8262      -0.0046
  test_rate_hike        0.8638      0.8719      -0.0081


In [ ]:
# ── §1.6 · Coefficient analysis (β-hat interpretation) ─────────────────────────
# Logistic regression coefficients are interpretable as log-odds contributions.
# Odds ratio = exp(β): a one-unit increase in the standardised feature multiplies
# the odds of delinquency by exp(β).
#
# Important: These are not causal estimates (Mullainathan & Spiess 2017).
# The model is a prediction tool; coefficients capture partial correlations
# conditional on all other features, not structural effects.
#
# Coefficients are sorted by |standardised coefficient| to identify the most
# predictive features.

coefs    = LOGIT.coef_[0]   # shape (n_features_total,)
feat_n   = np.array(FEAT_NAMES_ALL)

if len(coefs) != len(feat_n):
    min_len = min(len(coefs), len(feat_n))
    coefs  = coefs[:min_len]
    feat_n = feat_n[:min_len]

# Sort by absolute value of standardised coefficient (already standardised by pipeline)
order  = np.argsort(np.abs(coefs))[::-1]
top_n  = min(25, len(coefs))

coef_df = pd.DataFrame({
    "feature"   : feat_n[order[:top_n]],
    "coef"      : coefs[order[:top_n]],
    "odds_ratio": np.exp(coefs[order[:top_n]]),
})
coef_df["direction"] = np.where(coef_df["coef"] > 0, "↑ risk", "↓ risk")

print(f"=== Top {top_n} features by |standardised coefficient| ===")
print(f"{'Rank':<5} {'Feature':<38} {'Coef':>10}  {'Odds Ratio':>12}  Dir")
print("-" * 75)
for rank, (_, row) in enumerate(coef_df.iterrows(), 1):
    print(f"  {rank:<3}  {row['feature']:<38}  {row['coef']:>+10.4f}  {row['odds_ratio']:>12.4f}  {row['direction']}")

# Save coefficient table
coef_df_full = pd.DataFrame({
    "feature"   : feat_n,
    "coef"      : coefs,
    "odds_ratio": np.exp(coefs),
})
coef_df_full.to_csv(NB10_OUT_DIR / "nb10_coefficients.csv", index=False)
print(f"\n✓  Full coefficient table saved → nb10_coefficients.csv  ({len(coef_df_full)} rows)")

# ── Modeling-hazard diagnostics (computed and recorded) ──────────────────────
print("\n=== Coefficient pathology diagnostics ===")
COEF_DIAGNOSTICS: dict = {}
# (a) Bit-identical coefficients → perfectly collinear in the training window
_uniq, _inv, _cnt = np.unique(np.round(coefs, 12), return_inverse=True, return_counts=True)
_dups = [np.where(_inv == k)[0] for k, c in enumerate(_cnt) if c > 1]
COEF_DIAGNOSTICS["collinear_groups"] = [[str(x) for x in feat_n[g]] for g in _dups]
if _dups:
    print("  ⚠  Bit-identical coefficient groups (perfect collinearity in the fit window):")
    for g in _dups:
        print(f"       coef={coefs[g[0]]:+.6f}: {', '.join(feat_n[g])}")
else:
    print("  No bit-identical coefficient groups detected.")
# (b) CLTV vs LTV sign relationship (CLTV ≥ LTV by construction → near-collinear)
if "original_ltv" in feat_n and "original_cltv" in feat_n:
    _cl = float(coefs[np.where(feat_n == "original_ltv")[0][0]])
    _cc = float(coefs[np.where(feat_n == "original_cltv")[0][0]])
    COEF_DIAGNOSTICS["ltv_coef"] = _cl
    COEF_DIAGNOSTICS["cltv_coef"] = _cc
    COEF_DIAGNOSTICS["ltv_cltv_sign_flip"] = bool(np.sign(_cl) != np.sign(_cc))
    if np.sign(_cl) != np.sign(_cc):
        print(f"  ⚠  original_ltv ({_cl:+.4f}) and original_cltv ({_cc:+.4f}) carry OPPOSITE signs.")
        print(f"       CLTV ≥ LTV by construction (near-collinear), so the conditional sign flip is a")
        print(f"       collinearity artifact, not an economic effect - do not interpret in isolation.")
# (c) vintage_year as a standardized NUMERIC → out-of-window extrapolation hazard
if "vintage_year" in feat_n:
    _cv = float(coefs[np.where(feat_n == "vintage_year")[0][0]])
    COEF_DIAGNOSTICS["vintage_year_coef_per_sigma"] = _cv
    print(f"  ⚠  vintage_year enters as a standardized numeric (coef {_cv:+.4f}/σ), fit on 1999–2003.")
    print(f"       Test loans (2007–2023) sit many σ above the training mean, so their log-odds get a")
    print(f"       LINEARLY EXTRAPOLATED shift - a hazard absent from the LightGBM (categorical vintage)")
    print(f"       and a plausible contributor to the logit's uniformly higher PSI. Consider treating")
    print(f"       vintage_year categorically.")

=== Top 25 features by |standardised coefficient| ===
Rank  Feature                                      Coef    Odds Ratio  Dir
---------------------------------------------------------------------------
  1    credit_score                               -0.8338        0.4344  ↓ risk
  2    is_multi_borrower                          -0.3373        0.7137  ↓ risk
  3    loan_purpose__P                            -0.3189        0.7270  ↓ risk
  4    is_30dpd_at_t                              +0.3090        1.3620  ↑ risk
  5    original_upb                               -0.2930        0.7460  ↓ risk
  6    original_ltv                               +0.2837        1.3280  ↑ risk
  7    property_type__MH                          +0.2727        1.3135  ↑ risk
  8    property_type__SF                          +0.2423        1.2742  ↑ risk
  9    census_division__New England               -0.2263        0.7975  ↓ risk
  10   census_division__Pacific                   -0.2140        0.8073  ↓ 

**Coefficient-scope note.** Numeric predictors are standardised, whereas one-hot indicators remain 0/1, so `|β|` is not a uniform standardised feature-importance scale. `exp(β)` corresponds to a one-standard-deviation change for standardised numeric predictors and a 0→1 change for dummy indicators. The printed coefficient/collinearity diagnostics are descriptive only; coefficient equality or sign reversals do not establish collinearity. The vintage warning is benchmark-specific only in its linear extrapolation: LightGBM also operates outside its training-vintage support.

---
## Section 2 · Probability Calibration: Does Logit Need Isotonic Recalibration?

### Why this matters for CP

The CP nonconformity score `s(X, Y) = 1 - p̂(Y|X)` depends on the
**numerical probability scale**, not just the ranking. If `p̂` is
systematically biased (overconfident or underconfident), the threshold
quantile `q̂` will still achieve marginal coverage - CP's validity is a
finite-sample exchangeability result, not a calibration result (Vovk et al. 2005;
Papadopoulos et al. 2002) - but the prediction sets' *efficiency* (set size) and
class-conditional behaviour can degrade severely.

Isotonic recalibration substantially reduced the Brier score for LightGBM.

Logistic regression tends to be empirically well calibrated: it optimises the
log-loss with a canonical Bernoulli link function, and Niculescu-Mizil & Caruana
(2005) find that post-hoc calibration often provides little additional benefit for
logistic regression, in contrast to maximum-margin methods such as boosted trees
and SVMs, whose predictions show a characteristic sigmoid distortion. However, with `class_weight='balanced'` and L2 regularisation, calibration may degrade. This is tested empirically and isotonic recalibration is applied if needed for methodological consistency.

The CP-relevant question is whether isotonic recalibration materially changes the logit CP metrics. This is addressed in §6.


In [ ]:
# ── §2.1 · Raw logit calibration diagnostics ──────────────────────────────────
# Computed on the full 5% calibration benchmark sample (obs_year 2005–2006).
# The deterministic 10/90 iso_fit/cp_cal partition is defined in §2.2.

df_cal  = EVAL_RESULTS["calibration"]["df"]
p_raw   = EVAL_RESULTS["calibration"]["p_hat_raw"]
y_cal   = EVAL_RESULTS["calibration"]["y"]

# ── Raw Brier score and probability distribution diagnostics ──────────────────
brier_raw = float(brier_score_loss(y_cal, p_raw))
print(f"Raw logit calibration statistics (calibration split, 5 % sample):")
print(f"  n rows              : {len(y_cal):,}")
print(f"  positive rate       : {y_cal.mean()*100:.4f}%")
print(f"  Brier score (raw)   : {brier_raw:.6f}  (LightGBM pre-iso OOS: {LGBM_BRIER_PRE:.6f})")
print(f"  p_hat range         : [{p_raw.min():.6f}, {p_raw.max():.6f}]")
print(f"  mean p_hat          : {p_raw.mean():.6f}")
print(f"  median p_hat        : {np.median(p_raw):.6f}")

# Fraction of p_hat concentrated near 0 and 1 (bimodality indicator)
pct_below_001 = (p_raw < 0.01).mean() * 100
pct_above_099 = (p_raw > 0.99).mean() * 100
print(f"  p_hat < 0.01        : {pct_below_001:.2f}%")
print(f"  p_hat > 0.99        : {pct_above_099:.2f}%")
print()

# Reliability diagram data
frac_pos, mean_pred = calibration_curve(y_cal, p_raw, n_bins=10, strategy="quantile")
ece_raw = np.mean(np.abs(frac_pos - mean_pred))  # approximate ECE (uniform bin weight)
print(f"  Approx. ECE (raw)   : {ece_raw:.6f}")
print()
print("  Reliability diagram (bin mean_pred vs empirical positive rate):")
print(f"  {'Bin':>4}  {'mean_pred':>10}  {'empirical_pos':>14}  {'|diff|':>8}")
for i, (mp, fp) in enumerate(zip(mean_pred, frac_pos)):
    print(f"  {i+1:>4}  {mp:>10.4f}  {fp:>14.4f}  {abs(mp-fp):>8.4f}")

Raw logit calibration statistics (calibration split, 5 % sample):
  n rows              : 8,972,803
  positive rate       : 1.1031%
  Brier score (raw)   : 0.066746  (LightGBM pre-iso OOS: 0.064263)
  p_hat range         : [0.002035, 1.000000]
  mean p_hat          : 0.177285
  median p_hat        : 0.099957
  p_hat < 0.01        : 1.56%
  p_hat > 0.99        : 0.57%

  Approx. ECE (raw)   : 0.166254

  Reliability diagram (bin mean_pred vs empirical positive rate):
   Bin   mean_pred   empirical_pos    |diff|
     1      0.0152          0.0004    0.0148
     2      0.0291          0.0006    0.0285
     3      0.0438          0.0010    0.0428
     4      0.0619          0.0012    0.0607
     5      0.0856          0.0018    0.0838
     6      0.1179          0.0025    0.1154
     7      0.1640          0.0039    0.1602
     8      0.2348          0.0062    0.2286
     9      0.3547          0.0109    0.3438
    10      0.6658          0.0819    0.5838


In [ ]:
# ── §2.2 · Isotonic recalibration for logit ────────────────────────────────────
# Methodology identical to NB05a.
# iso_fit subset  : deterministic 10% loan-level CRC32 partition of calibration rows
#                   (crc32(loan_sequence_number) % 10 == 0); allocation is label-independent,
#                   not outcome-stratified.
# cp_cal subset   : the remaining ~90%.
#
# Isotonic via PAVA matches NB05a; calibration rationale for logit is in the §2 markdown.

# Deterministic, label-independent 10% loan-level iso-fit split.
_cal_lsn  = df_cal["loan_sequence_number"].cast(pl.Utf8).to_list()
iso_mask  = np.array([zlib.crc32(s.encode()) % 10 == 0 for s in _cal_lsn], dtype=bool)
n_iso     = int(iso_mask.sum())
n_cp_cal  = int((~iso_mask).sum())

p_iso_fit  = p_raw[iso_mask];    y_iso_fit  = y_cal[iso_mask]
p_cp_cal   = p_raw[~iso_mask];   y_cp_cal   = y_cal[~iso_mask]
df_cp_cal  = df_cal.filter(pl.Series("_iso_mask", ~iso_mask))  # Polars rows for cp_cal

print(f"iso_fit rows : {n_iso:,}  (pos rate {y_iso_fit.mean()*100:.4f}%)")
print(f"cp_cal rows  : {n_cp_cal:,}  (pos rate {y_cp_cal.mean()*100:.4f}%)")

# Fit PAVA isotonic regressor on iso_fit
ISO_CALIBRATOR_LOGIT = IsotonicRegression(out_of_bounds="clip")
ISO_CALIBRATOR_LOGIT.fit(p_iso_fit, y_iso_fit)

# Apply to cp_cal (OOS)
p_cp_cal_iso = ISO_CALIBRATOR_LOGIT.predict(p_cp_cal).astype(np.float32)

brier_raw_oos = float(brier_score_loss(y_cp_cal, p_cp_cal))
brier_iso_oos = float(brier_score_loss(y_cp_cal, p_cp_cal_iso))
auroc_raw     = roc_auc_score(y_cp_cal, p_cp_cal)
auroc_iso     = roc_auc_score(y_cp_cal, p_cp_cal_iso)

print(f"\nCalibration comparison (cp_cal OOS, {n_cp_cal:,} rows):")
print(f"  Brier  raw   : {brier_raw_oos:.6f}")
print(f"  Brier  iso   : {brier_iso_oos:.6f}  (improvement: {(brier_raw_oos-brier_iso_oos)/brier_raw_oos*100:.1f}%)")
print(f"  AUROC  raw   : {auroc_raw:.6f}")
print(f"  AUROC  iso   : {auroc_iso:.6f}  (delta: {auroc_iso - auroc_raw:+.6f})")
print()
print(f"  LightGBM reference (from NB05a):")
print(f"    Brier pre-iso (OOS): {LGBM_BRIER_PRE:.6f}  |  post-iso: {LGBM_BRIER_POST:.6f}  "
          f"(improvement: {LGBM_BRIER_IMPROVE_PCT:.1f}%)")
print(f"    AUROC delta (OOS)  : {LGBM_AUROC_DELTA:+.6f}  (rank ordering preserved)")
print()
print(f"  → Logit Brier improvement {(brier_raw_oos-brier_iso_oos)/brier_raw_oos*100:.1f}% vs LightGBM {LGBM_BRIER_IMPROVE_PCT:.1f}%:")
if abs(brier_iso_oos - brier_raw_oos) < 0.005:
    print("    Logit is better-calibrated natively -- isotonic correction is small.")
else:
    print("    Isotonic recalibration provides meaningful improvement for logit too.")

n_bp = int(len(np.unique(ISO_CALIBRATOR_LOGIT.y_thresholds_)))
print(f"  PAVA breakpoints         : {n_bp}  (LightGBM: {LGBM_PAVA_BP if LGBM_PAVA_BP is not None else 'not stored'})")

# Fraction of probabilities near extremes after iso-recalibration
p_iso_full = ISO_CALIBRATOR_LOGIT.predict(p_raw).astype(np.float32)
EVAL_RESULTS["calibration"]["p_hat_iso"] = p_iso_full
print(f"\n  p_hat_iso range : [{p_iso_full.min():.6f}, {p_iso_full.max():.6f}]")

# Also store iso-recalibrated probabilities for all test splits
for split in TEST_SPLITS:
    p_r = EVAL_RESULTS[split]["p_hat_raw"]
    EVAL_RESULTS[split]["p_hat_iso"] = ISO_CALIBRATOR_LOGIT.predict(p_r).astype(np.float32)

iso_fit rows : 892,325  (pos rate 1.1358%)
cp_cal rows  : 8,080,478  (pos rate 1.0994%)

Calibration comparison (cp_cal OOS, 8,080,478 rows):
  Brier  raw   : 0.066621
  Brier  iso   : 0.008923  (improvement: 86.6%)
  AUROC  raw   : 0.908676
  AUROC  iso   : 0.907697  (delta: -0.000980)

  LightGBM reference (from NB05a):
    Brier pre-iso (OOS): 0.064263  |  post-iso: 0.008616  (improvement: 86.6%)
    AUROC delta (OOS)  : -0.000060  (rank ordering preserved)

  → Logit Brier improvement 86.6% vs LightGBM 86.6%:
    Isotonic recalibration provides meaningful improvement for logit too.
  PAVA breakpoints         : 70  (LightGBM: 516)

  p_hat_iso range : [0.000000, 1.000000]


**Diagnostic note.** The preceding `516 PAVA breakpoints` label refers to the upstream
LightGBM calibrator imported as a reference. The logistic isotonic calibrator fitted in §2.2
has 70 retained breakpoints. This affects only the printed descriptive label; all logit
probabilities and downstream coverage calculations use the separately fitted logistic calibrator. Breakpoint counts are not directly comparable: NB10 reports unique logit isotonic levels, whereas NB05a counts interpolation knots. The later §6.2 `516 PAVA breakpoints` label is the imported LightGBM count, not a logit count; all logit probabilities and downstream results use `ISO_CALIBRATOR_LOGIT`.

In [ ]:
# ── §2.3 · Calibration comparison table ──────────────────────────────────
lgbm_cal_auroc = LGBM_DISC.get("calibration", {}).get("auroc", float("nan"))
lgbm_raw_auroc_approx = lgbm_cal_auroc + LGBM_AUROC_DELTA  # pre-iso ≈ post-iso + delta

print("=== Calibration Comparison: Raw Logit vs Iso-Recalibrated Logit vs LightGBM ===")
print()
print(f"{'Model/Variant':<35} {'Brier':>10}  {'ECE (approx)':>13}  {'AUROC':>8}  {'p<0.01 (%)':>10}")
print("-" * 82)

for label, p_use in [("Logit (raw, class_weight=balanced)", p_raw),
                     ("Logit (isotonic-recalibrated)",       p_iso_full)]:
    brier  = float(brier_score_loss(y_cal, p_use))
    fp_b, mp_b = calibration_curve(y_cal, p_use, n_bins=10, strategy="quantile")
    ece    = float(np.mean(np.abs(fp_b - mp_b)))
    auroc  = roc_auc_score(y_cal, p_use)
    below  = (p_use < 0.01).mean() * 100
    print(f"  {label:<33}  {brier:>10.6f}  {ece:>13.6f}  {auroc:>8.6f}  {below:>10.2f}%")

print(f"  {'LightGBM (raw)':<33}  {LGBM_BRIER_PRE:>10.6f}  {'N/A':>13}  {'N/A':>8}  {'N/A':>10}")
print(f"  {'LightGBM (isotonic-recalibrated)':<33}  {LGBM_BRIER_POST:>10.6f}  {'N/A':>13}  {'N/A':>8}  {'N/A':>10}")

cal_comp_df = pd.DataFrame([
    {"model":"logit_raw", "brier": float(brier_score_loss(y_cal, p_raw)),
     "auroc": float(roc_auc_score(y_cal, p_raw))},
    {"model":"logit_iso", "brier": float(brier_score_loss(y_cal, p_iso_full)),
     "auroc": float(roc_auc_score(y_cal, p_iso_full))},
    {"model":"lgbm_raw",  "brier": LGBM_BRIER_PRE,  "auroc": lgbm_raw_auroc_approx},
    {"model":"lgbm_iso",  "brier": LGBM_BRIER_POST, "auroc": lgbm_cal_auroc},
])
cal_comp_df.to_csv(NB10_OUT_DIR / "nb10_calibration_comparison.csv", index=False)

=== Calibration Comparison: Raw Logit vs Iso-Recalibrated Logit vs LightGBM ===

Model/Variant                            Brier   ECE (approx)     AUROC  p<0.01 (%)
----------------------------------------------------------------------------------
  Logit (raw, class_weight=balanced)    0.066746       0.166254  0.908869        1.56%
  Logit (isotonic-recalibrated)        0.008958       0.000386  0.908180       82.20%
  LightGBM (raw)                       0.064263            N/A       N/A         N/A
  LightGBM (isotonic-recalibrated)     0.008616            N/A       N/A         N/A


---
## Section 3 · CP Threshold Calibration (Logit)

All CP thresholds are computed on the **iso-recalibrated logit probabilities** using the same finite-sample formula as NB05b:

$$\hat{q} = \text{(}\lceil (n+1)(1-\alpha) \rceil\text{)-th order statistic of calibration nonconformity scores}$$

Under exchangeability between calibration and test observations, this split-CP quantile
choice gives the usual finite-sample marginal coverage guarantee for the resulting
prediction set. With finite calibration samples and deterministic tie handling, the
coverage is conventionally described as at least $(1-\alpha)$, up to the standard
finite-sample discreteness of conformal quantiles (Papadopoulos et al., 2002; Vovk
et al., 2005).

**Score definitions (identical to NB05a/05b):**
- `score_scp(x,y) = 1 - p̂(y|x)` - standard split-CP nonconformity score
- `score_aps(x,y)` - Generalised Inverse Quantile (GIQ) score (Romano, Sesia & Candès 2020)

The cp_cal set used here is the 90 % portion of the 5 % calibration sample not used for iso_fit (`p_cp_cal_iso`, `y_cp_cal`, `df_cp_cal` constructed in §2.2).

**Diagnostic-label note.** The stored §3.1 and §3.3 lines labelled “Empty-set rate” are actually the true-label **miscoverage rate**, `P(s_true > q̂)`. Full empty-set rates are constructed in §4; these calibration-anchor printouts should therefore not be read as set-composition statistics.


In [ ]:
def cp_quantile(scores: np.ndarray, alpha: float = ALPHA) -> np.float32:
    """
    Finite-sample split-CP threshold: the ceil((n+1)(1-alpha))-th order statistic.
    Uses np.partition for O(n) order-statistic selection (introselect).
    Returns np.float32(np.inf) when the adjusted rank exceeds n, per the strict
    conformal convention (boundary case: alpha < 1/(n+1)).

    Reference: Lei et al. (2018) for split-conformal finite-sample coverage.
    """
    n = len(scores)
    k = int(np.ceil((n + 1) * (1.0 - alpha)))
    if k > n:
        return np.float32(np.inf)          # strict conformal boundary: all test points covered
    idx = k - 1                            # convert 1-indexed rank to 0-indexed
    return np.float32(np.partition(scores, idx)[idx])

def compute_score_scp(p_hat: np.ndarray, y: np.ndarray) -> np.ndarray:
    """
    SCP nonconformity score: s(x,y) = 1 - p_hat_cal(true class | x).
    For y=0: s = p_hat  (1 - P(Y=0|X) = P(Y=1|X) = p_hat)
    For y=1: s = 1 - p_hat
    Reference: Sadinle, Lei & Wasserman (2019) for the set-valued (LAC) score.
    """
    return np.where(y == 1, 1.0 - p_hat, p_hat).astype(np.float32)

def compute_score_aps(p_hat: np.ndarray, y: np.ndarray, U: np.ndarray) -> np.ndarray:
    """
    Generalised Inverse Quantile (GIQ) nonconformity score for binary classification.
    s(x,y) = sum_{k: P(Y=k|x) > P(Y=y|x)} P(Y=k|x) + (1 - U) * P(Y=y|x)

    For y=1: if p >= 0.5 (class 1 most probable): s = (1-U)*p
             if p < 0.5  (class 0 more probable):  s = (1-p) + (1-U)*p
    For y=0: if (1-p) >= p (class 0 most probable): s = (1-U)*(1-p)
             if (1-p) < p  (class 1 more probable): s = p + (1-U)*(1-p)

    U ~ Uniform[0,1] is a per-observation random draw used in the randomized APS/GIQ
    score. It reduces discreteness/tie problems in the conformity scores and matches the
    randomized score construction in the cited APS/GIQ method.

    Reference: Romano, Sesia & Candès (2020).
    The alternative p_above + U·p_y is distributionally equivalent (U and 1-U are
    identically Uniform[0,1]) but is not the cited form. This notebook matches NB05a
    exactly to ensure score-file reproducibility across the pipeline.
    """
    p1 = p_hat.astype(np.float32)
    p0 = (1.0 - p_hat).astype(np.float32)
    U  = U.astype(np.float32)
    return np.where(
        y == 1,
        np.where(p1 >= p0, (1.0 - U) * p1,         p0 + (1.0 - U) * p1),   # true label is 1
        np.where(p0 >= p1, (1.0 - U) * p0,         p1 + (1.0 - U) * p0),   # true label is 0
    ).astype(np.float32)

def batch_seed(split: str, year: int) -> int:
    """Deterministic batch seed from (split, year) via MD5 - logit namespace."""
    return int(hashlib.md5(f"logit_{split}_{year}".encode()).hexdigest()[:8], 16)

print("✓  CP helper functions defined (cp_quantile, compute_score_scp, compute_score_aps)")

# ── Compute SCP threshold on cp_cal ───────────────────────────────────────────
scores_scp_cp_cal = compute_score_scp(p_cp_cal_iso, y_cp_cal)
Q_HAT_SCP_LOGIT   = cp_quantile(scores_scp_cp_cal, ALPHA)

print(f"\nSCP threshold (logit):  q_hat = {Q_HAT_SCP_LOGIT:.8f}")
print(f"SCP threshold (LightGBM): q_hat = {LGBM_Q_SCP:.8f}  (ratio: {float(Q_HAT_SCP_LOGIT)/LGBM_Q_SCP:.4f}×)")
print(f"n_cp_cal = {len(y_cp_cal):,}")

# In-sample class-conditional coverage
cov_mc   = float((scores_scp_cp_cal <= Q_HAT_SCP_LOGIT).mean())
cov_y0   = float((scores_scp_cp_cal[y_cp_cal==0] <= Q_HAT_SCP_LOGIT).mean())
cov_y1   = float((scores_scp_cp_cal[y_cp_cal==1] <= Q_HAT_SCP_LOGIT).mean())
empty_rt = float((scores_scp_cp_cal >  Q_HAT_SCP_LOGIT).mean())

print(f"\nSCP in-sample cp_cal coverage (logit):")
print(f"  Marginal        : {cov_mc*100:.4f}%  (target 90.00%)")
print(f"  Coverage(y=0)   : {cov_y0*100:.4f}%")
print(f"  Coverage(y=1)   : {cov_y1*100:.6f}%  (LightGBM SCP cp_cal: {LGBM_SCP_COV_Y1_CAL*100:.4f}%)")
print(f"  Empty-set rate  : {empty_rt*100:.4f}%")
print()
print(f"  Score distribution:")
print(f"    mean(score_scp | y=0) : {scores_scp_cp_cal[y_cp_cal==0].mean():.6f}")
print(f"    mean(score_scp | y=1) : {scores_scp_cp_cal[y_cp_cal==1].mean():.6f}")
print(f"    q_hat_SCP             : {float(Q_HAT_SCP_LOGIT):.6f}")
print(f"    Pct(y=1 scores > q)   : {(scores_scp_cp_cal[y_cp_cal==1] > Q_HAT_SCP_LOGIT).mean()*100:.2f}%")

✓  CP helper functions defined (cp_quantile, compute_score_scp, compute_score_aps)

SCP threshold (logit):  q_hat = 0.01836919
SCP threshold (LightGBM): q_hat = 0.01622366  (ratio: 1.1322×)
n_cp_cal = 8,080,478

SCP in-sample cp_cal coverage (logit):
  Marginal        : 92.9354%  (target 90.00%)
  Coverage(y=0)   : 93.9674%
  Coverage(y=1)   : 0.102431%  (LightGBM SCP cp_cal: 0.0039%)
  Empty-set rate  : 7.0646%

  Score distribution:
    mean(score_scp | y=0) : 0.009153
    mean(score_scp | y=1) : 0.811642
    q_hat_SCP             : 0.018369
    Pct(y=1 scores > q)   : 99.90%


In [ ]:
# ── §3.2 · Mondrian FICO×LTV thresholds (logit) ────────────────────────────────
# Per-group finite-sample quantiles on group-specific SCP nonconformity scores.
# Groups with < 1000 cp_cal rows are assigned the global SCP threshold as fallback.
# Note: the <1000 minimum-cell rule is an engineering stability choice in this
# notebook.

MONDRIAN_MIN_ROWS = 1000

groups_cp_cal = df_cp_cal["fico_ltv_group"].to_numpy()
unique_cells  = np.unique(groups_cp_cal)

MONDRIAN_LOGIT: dict[str, dict] = {}

print(f"{'Cell':<30}  {'q_hat':>10}  {'ratio/SCP':>10}  {'n_rows':>8}  {'cov':>8}")
print("-" * 75)

for cell in sorted(unique_cells):
    mask    = groups_cp_cal == cell
    n_rows  = mask.sum()
    if n_rows < MONDRIAN_MIN_ROWS:
        q = float(Q_HAT_SCP_LOGIT)
        note = " [fallback to global SCP]"
    else:
        q = float(cp_quantile(scores_scp_cp_cal[mask], ALPHA))
        note = ""
    cov = float((scores_scp_cp_cal[mask] <= np.float32(q)).mean())
    MONDRIAN_LOGIT[cell] = {"q_hat": q, "n_rows": int(n_rows), "in_sample_cov": cov}
    ratio = q / float(Q_HAT_SCP_LOGIT) if float(Q_HAT_SCP_LOGIT) > 0 else float("nan")
    print(f"  {cell:<28}  {q:>10.6f}  {ratio:>10.2f}×  {n_rows:>8,}  {cov:>8.4f}{note}")

vals   = [v["q_hat"] for v in MONDRIAN_LOGIT.values() if v["n_rows"] >= MONDRIAN_MIN_ROWS]
spread = max(vals) / min(vals) if min(vals) > 0 else float("nan")
print(f"\n  FICO×LTV threshold spread (logit)    : {spread:.2f}×")
print(f"  FICO×LTV threshold spread (LightGBM) : {LGBM_MONDRIAN_SPREAD:.2f}×")
print(f"  → {'Logit spread is LARGER' if spread > LGBM_MONDRIAN_SPREAD else 'Logit spread is SMALLER'} than LightGBM spread.")

Cell                                 q_hat   ratio/SCP    n_rows       cov
---------------------------------------------------------------------------
  Near-Prime × High               0.101629        5.53×    24,954    0.9102
  Near-Prime × Low                0.018369        1.00×  2,131,553    0.9204
  Near-Prime × Moderate           0.056606        3.08×   429,243    0.9080
  Prime × High                    0.012526        0.68×    22,235    0.9132
  Prime × Low                     0.005017        0.27×  4,545,674    0.9396
  Prime × Moderate                0.010820        0.59×   388,737    0.9006
  Sentinel                        0.018369        1.00×    27,893    0.9059
  Subprime × High                 0.416884       22.69×     4,382    0.9062
  Subprime × Low                  0.191659       10.43×   398,118    0.9006
  Subprime × Moderate             0.416884       22.69×   107,689    0.9025

  FICO×LTV threshold spread (logit)    : 83.10×
  FICO×LTV threshold spread (LightGBM)

In [ ]:
# ── §3.3 · APS threshold (logit) ───────────────────────────────────────────────
# APS uses the same finite-sample quantile formula, applied to GIQ scores.
# Seed uses no year-specific suffix; cp_cal rows are not sorted by year.
# Deterministic U drawn per cp_cal batch using MD5-based seed (logit namespace).

# Generate deterministic U for cp_cal rows
rng_cp   = np.random.default_rng(seed=batch_seed("calibration_cp_cal", 0))
U_cp_cal = rng_cp.uniform(size=len(y_cp_cal)).astype(np.float32)

scores_aps_cp_cal = compute_score_aps(p_cp_cal_iso, y_cp_cal, U_cp_cal)
Q_HAT_APS_LOGIT   = cp_quantile(scores_aps_cp_cal, ALPHA)

print(f"APS threshold (logit)    : q_hat = {Q_HAT_APS_LOGIT:.8f}")
print(f"APS threshold (LightGBM) : q_hat = {LGBM_Q_APS:.8f}  (ratio: {float(Q_HAT_APS_LOGIT)/LGBM_Q_APS:.6f}×)")

cov_mc_aps  = float((scores_aps_cp_cal <= Q_HAT_APS_LOGIT).mean())
cov_y0_aps  = float((scores_aps_cp_cal[y_cp_cal==0] <= Q_HAT_APS_LOGIT).mean())
cov_y1_aps  = float((scores_aps_cp_cal[y_cp_cal==1] <= Q_HAT_APS_LOGIT).mean())
empty_aps   = float((scores_aps_cp_cal >  Q_HAT_APS_LOGIT).mean())

print(f"\nAPS in-sample cp_cal coverage (logit):")
print(f"  Marginal       : {cov_mc_aps*100:.6f}%  (target 90.00%)")
print(f"  Coverage(y=0)  : {cov_y0_aps*100:.4f}%")
print(f"  Coverage(y=1)  : {cov_y1_aps*100:.4f}%  (LightGBM APS cp_cal: {LGBM_APS_COV_Y1_CAL*100:.4f}%)")
print(f"  Empty-set rate : {empty_aps*100:.4f}%")
print()
print(f"  Improvement APS vs SCP in Cov(y=1):")
delta_pp = (cov_y1_aps - cov_y1) * 100
print(f"    Logit   : SCP {cov_y1*100:.4f}% → APS {cov_y1_aps*100:.4f}% (+{delta_pp:.2f} pp)")
print(f"    LightGBM: SCP  {LGBM_SCP_COV_Y1_CAL*100:.4f}%  → APS {LGBM_APS_COV_Y1_CAL*100:.4f}%"      f"  (+{(LGBM_APS_COV_Y1_CAL - LGBM_SCP_COV_Y1_CAL)*100:.2f} pp)")

APS threshold (logit)    : q_hat = 0.89983147
APS threshold (LightGBM) : q_hat = 0.89995086  (ratio: 0.999867×)

APS in-sample cp_cal coverage (logit):
  Marginal       : 90.000022%  (target 90.00%)
  Coverage(y=0)  : 90.6338%
  Coverage(y=1)  : 32.9896%  (LightGBM APS cp_cal: 37.6046%)
  Empty-set rate : 10.0000%

  Improvement APS vs SCP in Cov(y=1):
    Logit   : SCP 0.1024% → APS 32.9896% (+32.89 pp)
    LightGBM: SCP  0.0039%  → APS 37.6046%  (+37.60 pp)


In [ ]:
# ── §3.4 · Calibration diagnostics for logit CP ────────────────────────────────

n_pass = 0; n_fail = 0
failed_tests: list = []

def check(cond, name, tol_msg=""):
    global n_pass, n_fail
    if cond:
        print(f"  ✓  {name}")
        n_pass += 1
    else:
        print(f"  ✗  FAIL: {name}  {tol_msg}")
        n_fail += 1
        failed_tests.append(name)

TOL = 0.005  # 0.5 pp tolerance on in-sample coverage

check(abs(cov_mc - (1.0 - ALPHA)) < TOL, f"SCP marginal coverage = {cov_mc:.4f}")
check(cov_y1 < 0.01,
      f"SCP class-conditional collapse: Cov(y=1) < 1%  (actual: {cov_y1*100:.4f}%)")
check(abs(cov_mc_aps - (1.0 - ALPHA)) < TOL, f"APS marginal coverage = {cov_mc_aps:.6f}")
check(cov_y1_aps > 0.05,
      f"APS Cov(y=1) > 5% (actual: {cov_y1_aps*100:.4f}%)")

# Mondrian in-sample coverage within tolerance for cells with n >= MONDRIAN_MIN_ROWS
for cell, info in MONDRIAN_LOGIT.items():
    if info["n_rows"] >= MONDRIAN_MIN_ROWS:
        check(abs(info["in_sample_cov"] - (1.0-ALPHA)) < 0.02,
              f"Mondrian [{cell}] coverage = {info['in_sample_cov']:.4f}")

# Float32 round-trip
q_scp_f32 = np.float32(Q_HAT_SCP_LOGIT)
q_aps_f32 = np.float32(Q_HAT_APS_LOGIT)
check(float(q_scp_f32) == float(Q_HAT_SCP_LOGIT), "Float32 round-trip: q_hat_SCP_logit")
check(float(q_aps_f32) == float(Q_HAT_APS_LOGIT), "Float32 round-trip: q_hat_APS_logit")

print(f"\nUnit tests: {n_pass} passed / {n_fail} failed")

  ✗  FAIL: SCP marginal coverage = 0.9294  
  ✓  SCP class-conditional collapse: Cov(y=1) < 1%  (actual: 0.1024%)
  ✓  APS marginal coverage = 0.900000
  ✓  APS Cov(y=1) > 5% (actual: 32.9896%)
  ✓  Mondrian [Near-Prime × High] coverage = 0.9102
  ✗  FAIL: Mondrian [Near-Prime × Low] coverage = 0.9204  
  ✓  Mondrian [Near-Prime × Moderate] coverage = 0.9080
  ✓  Mondrian [Prime × High] coverage = 0.9132
  ✗  FAIL: Mondrian [Prime × Low] coverage = 0.9396  
  ✓  Mondrian [Prime × Moderate] coverage = 0.9006
  ✓  Mondrian [Sentinel] coverage = 0.9059
  ✓  Mondrian [Subprime × High] coverage = 0.9062
  ✓  Mondrian [Subprime × Low] coverage = 0.9006
  ✓  Mondrian [Subprime × Moderate] coverage = 0.9025
  ✓  Float32 round-trip: q_hat_SCP_logit
  ✓  Float32 round-trip: q_hat_APS_logit

Unit tests: 13 passed / 3 failed


**Validation note.** The three printed `FAIL` lines are overcoverage relative to closeness-to-90% tolerances, not conformal-validity or pipeline failures; this diagnostic block does not gate downstream execution.

---
## Section 4 · Coverage Evaluation across Regimes (Logit)

SCP, Mondrian FICO×LTV, and APS are evaluated on the 5 % hash-deterministic test-split samples. Results are compared directly against LightGBM in section 5.

**Metrics computed:**
- `MC` - marginal coverage; deviation from 90 % nominal
- `Cov(y=0)`, `Cov(y=1)` - class-conditional coverage  
- `CovGap` - max|MC_k - 90%| over FICO×LTV cells k
- `WGC` - worst-group coverage
- Empty-set rate (abstention signal)

Clopper–Pearson intervals are reported as **binomial reference intervals**. Repeated loan-months and overlapping 12-month labels violate the independent-binomial model, so these intervals understate dependence-driven uncertainty; as in the thesis, the ±2 percentage-point band, not interval exclusion, determines materiality.


In [ ]:
# ── §4.1 · Static evaluation loop (SCP, Mondrian, APS) ──────────────────────────
# Processes each test split on the pre-loaded 5 % sample.
# Isotonic-recalibrated logit probabilities (p_hat_iso) are used throughout.

METHODS    = ["scp", "mondrian_fico_ltv", "aps"]
COV_LOGIT: dict = {}   # coverage metrics keyed by [split][method]

for split in TEST_SPLITS:
    t0       = time.time()
    df_sp    = EVAL_RESULTS[split]["df"]
    p_iso    = EVAL_RESULTS[split]["p_hat_iso"]
    y_sp     = EVAL_RESULTS[split]["y"]
    n_sp     = len(y_sp)
    pos_mask = y_sp == 1
    neg_mask = ~pos_mask
    groups   = df_sp["fico_ltv_group"].to_numpy()

    # Deterministic U for APS (logit namespace)
    obs_years = df_sp["obs_year"].to_numpy()
    U_sp = np.empty(n_sp, dtype=np.float32)
    for yr in np.unique(obs_years):
        m = obs_years == yr
        rng_yr = np.random.default_rng(seed=batch_seed(split, int(yr)))
        U_sp[m] = rng_yr.uniform(size=m.sum()).astype(np.float32)

    # Compute nonconformity scores
    s_scp = compute_score_scp(p_iso, y_sp)
    s_aps = compute_score_aps(p_iso, y_sp, U_sp)

    COV_LOGIT[split] = {}
    for method in METHODS:
        if method == "scp":
            scores = s_scp
            thresh_fn = lambda g: np.float32(Q_HAT_SCP_LOGIT)
        elif method == "mondrian_fico_ltv":
            scores = s_scp
            def thresh_fn(g, _MOND=MONDRIAN_LOGIT):
                return np.float32(_MOND.get(g, {"q_hat": float(Q_HAT_SCP_LOGIT)})["q_hat"])
        else:  # aps
            scores = s_aps
            thresh_fn = lambda g: np.float32(Q_HAT_APS_LOGIT)

        # Covered indicator (vectorised)
        if method == "mondrian_fico_ltv":
            q_per_obs = np.array([thresh_fn(g) for g in groups], dtype=np.float32)
            covered   = scores <= q_per_obs
        else:
            q_global  = thresh_fn(None)
            covered   = scores <= q_global

        mc       = float(covered.mean())
        cov_y0_m = float(covered[neg_mask].mean()) if neg_mask.sum() > 0 else float("nan")
        cov_y1_m = float(covered[pos_mask].mean()) if pos_mask.sum() > 0 else float("nan")

        # ── Empty-set rate: P(C(x) = ∅) ≠ 1 - MC in general ─────────────────────────
        # 1-MC = P(true label not covered) includes wrong singleton predictions ({other_label}).

        # SCP: empty set when q < p_hat < 1-q  (both candidate-label scores > threshold)
        if method == "scp":
            empty_m = float(((p_iso > float(q_global)) &
                            (p_iso < (1.0 - float(q_global)))).mean())
        elif method == "mondrian_fico_ltv":
            empty_m = float(((p_iso > q_per_obs) &
                            (p_iso < (1.0 - q_per_obs))).mean())
        else:  # aps
            # Auxiliary APS set-composition reconstruction.
            # Standard randomized APS uses one shared U per observation for the full set.
            # Following NB06's auxiliary convention, this block uses a fresh U_alt
            # for the alternate label, so it is not the exact shared-U realization.
            # True-label coverage above is unaffected by this reconstruction.
            U_alt = np.empty(n_sp, dtype=np.float32)
            for yr in np.unique(obs_years):
                m_yr = obs_years == yr
                rng_alt = np.random.default_rng(
                    seed=batch_seed(split, int(yr)) ^ 0x80000000)
                U_alt[m_yr] = rng_alt.uniform(size=m_yr.sum()).astype(np.float32)
            y_alt  = (1 - y_sp).astype(np.int32)
            s_alt  = compute_score_aps(p_iso, y_alt, U_alt)
            empty_m = float(((s_aps > q_global) & (s_alt > q_global)).mean())

        # CovGap: max|MC_cell - (1-α)| over FICO×LTV cells (NB06 formula).
        # Cells with fewer than COVGAP_MIN_CELL_N rows are dropped for stability;
        # all remaining cells (including the missing-FICO/LTV "Sentinel" cell) are
        # retained, matching the NB06 LightGBM CovGap / WGC reference cell set.
        cell_coverages = {}
        cell_n = {}
        for cell in np.unique(groups):
            mask_c = groups == cell
            n_c = int(mask_c.sum())
            if n_c > 0:
                cell_coverages[cell] = float(covered[mask_c].mean())
                cell_n[cell] = n_c
        valid_covs = [v for k, v in cell_coverages.items()
                      if cell_n[k] >= COVGAP_MIN_CELL_N]
        covgap_pp = (max(abs(v - NOMINAL_COV) for v in valid_covs) * 100
             if valid_covs else float("nan"))
        wgc       = min(valid_covs) * 100 if valid_covs else float("nan")

        COV_LOGIT[split][method] = {
            "n": n_sp, "n_pos": int(pos_mask.sum()), "n_neg": int(neg_mask.sum()),
            "mc": mc, "cov_y0": cov_y0_m, "cov_y1": cov_y1_m,
            "empty_rate": empty_m, "dev_pp": (mc - NOMINAL_COV) * 100,
            "covgap_pp": covgap_pp, "wgc_pct": wgc,
            "cell_coverages": cell_coverages,
        }

    print(f"  {split:<18}  ({time.time()-t0:.0f}s)")

print("\n✓  Static evaluation complete.")

  test_subprime       (113s)
  test_normal         (143s)
  test_covid          (34s)
  test_rate_hike      (49s)

✓  Static evaluation complete.


In [ ]:
# ── §4.2 · Global coverage metrics table (logit) ─────────────────────────────────

print("Global metrics by method and split (Logit):")
print(f"  {'Split':<20}  {'Method':<22}  {'MC':>7}  {'Dev(pp)':>8}  {'Cov(y=1)%':>10}  {'Empty%':>8}  {'CovGap':>8}")
print("  " + "-"*90)

for split in TEST_SPLITS:
    for method in METHODS:
        m = COV_LOGIT[split][method]
        print(f"  {SPLIT_LABELS[split][:20]:<20}  {METH_LABELS[method]:<22}"
              f"  {m['mc']*100:>7.2f}  {m['dev_pp']:>+8.2f}"
              f"  {m['cov_y1']*100:>10.4f}  {m['empty_rate']*100:>8.2f}"
              f"  {m['covgap_pp']:>8.2f}")
    print()

Global metrics by method and split (Logit):
  Split                 Method                       MC   Dev(pp)   Cov(y=1)%    Empty%    CovGap
  ------------------------------------------------------------------------------------------
  Subprime (2007–12)    SCP                       91.08     +1.08      0.3301      7.89     76.54
  Subprime (2007–12)    Mondrian (FICO×LTV)       90.33     +0.33      0.6013      8.48      7.29
  Subprime (2007–12)    APS                       89.04     -0.96     27.7337      9.19      2.96

  Normal (2013–Sep 201  SCP                       93.84     +3.84      0.8036      5.48     69.85
  Normal (2013–Sep 201  Mondrian (FICO×LTV)       94.09     +4.09      1.1176      5.16      6.20
  Normal (2013–Sep 201  APS                       89.51     -0.49     28.9692      9.41      2.43

  COVID (2020–21)       SCP                       94.29     +4.29      0.1351      4.76     70.04
  COVID (2020–21)       Mondrian (FICO×LTV)       93.41     +3.41      0.2232

In [ ]:
# ── §4.3 · CovGap table (logit) ──────────────────────────────────────────

print("=== CovGap (pp) -- Logit ===")
print(f"{'Method':<22}  {'Subprime':>10}  {'Normal':>10}  {'COVID':>10}  {'RateHike':>10}")
for method in METHODS:
    row = "  " + f"{METH_LABELS[method]:<20}  "
    for split in TEST_SPLITS:
        row += f"  {COV_LOGIT[split][method]['covgap_pp']:>10.2f}"
    print(row)

print()
print("=== CovGap (pp) -- LightGBM (from NB07) ===")
print(f"{'Method':<22}  {'Subprime':>10}  {'Normal':>10}  {'COVID':>10}  {'RateHike':>10}")
lgbm_covgap = {"scp": LGBM_REF["covgap_scp"], "mondrian_fico_ltv": LGBM_REF["covgap_mond"], "aps": LGBM_REF["covgap_aps"]}
for method in METHODS:
    row = "  " + f"{METH_LABELS[method]:<20}  "
    for split in TEST_SPLITS:
        row += f"  {lgbm_covgap[method][split]:>10.2f}"
    print(row)

=== CovGap (pp) -- Logit ===
Method                    Subprime      Normal       COVID    RateHike
  SCP                          76.54       69.85       70.04       67.48
  Mondrian (FICO×LTV)           7.29        6.20        4.79        5.65
  APS                           2.96        2.43        2.69        2.76

=== CovGap (pp) -- LightGBM (from NB07) ===
Method                    Subprime      Normal       COVID    RateHike
  SCP                          79.69       47.07       31.97       24.94
  Mondrian (FICO×LTV)           5.26        7.34        5.04        6.15
  APS                           3.92        1.28        1.72        1.57


In [ ]:
# ── §4.4 · Regime diagnostics: PSI and mean score shift ────────────────────────────

def compute_psi(scores_base: np.ndarray, scores_test: np.ndarray,
                n_bins: int = 10, eps: float = 1e-8) -> float:
    """
    Population Stability Index (PSI) between two score distributions.
    Bin boundaries are equal-frequency quantiles of scores_base.
    PSI = Σ_b (p_base_b - p_test_b) * ln(p_base_b / p_test_b)
    Reference: Yurdakul & Naranjo (2020) for PSI threshold
    interpretation and statistical properties of the 0.10/0.25 rule-of-thumb cutoffs.
    """
    bin_edges = np.quantile(scores_base, np.linspace(0, 1, n_bins + 1))
    bin_edges[0]  = -np.inf
    bin_edges[-1] =  np.inf
    p_base = np.histogram(scores_base, bins=bin_edges)[0] / len(scores_base) + eps
    p_test = np.histogram(scores_test, bins=bin_edges)[0] / len(scores_test) + eps
    return float(np.sum((p_base - p_test) * np.log(p_base / p_test)))

# SCP nonconformity scores for cp_cal baseline and test splits
scores_base = scores_scp_cp_cal  # logit cp_cal scores (90 % of 5 % calibration sample)
mean_base   = float(scores_base.mean())

print(f"Mechanism diagnostics (Logit -- SCP nonconformity score distributions):")
print(f"  Reference: cp_cal mean_score_scp = {mean_base:.6f}")
print()

print(f"  {'Split':<20}  {'PSI':>8}  {'PSI band':>12}  {'Mean shift%':>12}  "
      f"{'Mechanism (a-priori)':<22}  {'Pos_rate%':>10}  {'SCP MC dev':>12}")
print("  " + "-"*108)

PSI_LOGIT: dict = {}
for split in TEST_SPLITS:
    p_iso_t = EVAL_RESULTS[split]["p_hat_iso"]
    y_t     = EVAL_RESULTS[split]["y"]
    s_test  = compute_score_scp(p_iso_t, y_t)
    psi     = compute_psi(scores_base, s_test)
    shift   = (s_test.mean() - mean_base) / (mean_base + 1e-10) * 100
    pos_rt  = y_t.mean() * 100
    mc_dev  = COV_LOGIT[split]["scp"]["dev_pp"]
    mech    = MECH_LABELS[split]
    band    = psi_band(psi)
    PSI_LOGIT[split] = {"psi": psi, "mean_shift_pct": shift, "pos_rate_pct": pos_rt, "psi_band": band}
    print(f"  {split:<20}  {psi:>8.4f}  {band:>12}  {shift:>+12.2f}%  "
          f"{mech:<22}  {pos_rt:>10.4f}%  {mc_dev:>+12.4f} pp")

print("\n  Note: the Mechanism column is the A-PRIORI taxonomy inherited from the LightGBM")
print("  classification (MECH_LABELS), not re-derived from the logit's own PSI. Where the logit")
print("  PSI band disagrees with the label (e.g. test_normal in the 'moderate' band vs the")
print("  inherited 'Stable' label), the label is the upstream hypothesis, not a logit finding.")

print()
print("  LightGBM PSI reference:")
for split in TEST_SPLITS:
    print(f"    {split:<20}  PSI={LGBM_REF['psi_scp'][split]:.4f}")

Mechanism diagnostics (Logit -- SCP nonconformity score distributions):
  Reference: cp_cal mean_score_scp = 0.017976

  Split                      PSI      PSI band   Mean shift%  Mechanism (a-priori)     Pos_rate%    SCP MC dev
  ------------------------------------------------------------------------------------------------------------
  test_subprime           0.0084         small        +75.12%  Tail Shift                  2.5317%       +1.0780 pp
  test_normal             0.1171      moderate        +16.74%  Stable                      1.5274%       +3.8437 pp
  test_covid              0.1669      moderate        +23.97%  Structural Shift            1.6672%       +4.2933 pp
  test_rate_hike          0.2565   substantial         -9.95%  Composition Shift           1.0403%       +5.8044 pp

  Note: the Mechanism column is the A-PRIORI taxonomy inherited from the LightGBM
  classification (MECH_LABELS), not re-derived from the logit's own PSI. Where the logit
  PSI band disagrees wi

**Regime-label note.** `MECH_LABELS` stores the thesis's **a priori regime hypotheses**; the labels are not learned from LightGBM or from the logit PSI. The stored names `Subprime` and `Stable` correspond to the thesis labels `Crisis` and `Stable drift`. The stored PSI terms `small/moderate/substantial` use the same cutoffs as the thesis terms `negligible/investigation/action`.

In [ ]:
# ── §4.5 · Statistical reporting: Clopper-Pearson exact 95 % CIs ─────────────────
# Row-level Clopper-Pearson intervals are very narrow at these sample sizes,
# but repeated loan-months and overlapping forward labels violate independence.
# As in the thesis, treat them as scale references; ±2 pp defines materiality.

def clopper_pearson(n_cov: int, n_tot: int, alpha_ci: float = 0.05) -> tuple[float, float]:
    """Exact Clopper-Pearson binomial confidence interval with boundary handling.

    Boundary cases (n_cov == 0 or n_cov == n_tot) are handled explicitly
    because stats.beta.ppf returns NaN when a shape parameter is zero.
    """
    if n_tot <= 0:
        return float("nan"), float("nan")
    if n_cov < 0 or n_cov > n_tot:
        raise ValueError(f"n_cov must satisfy 0 <= n_cov <= n_tot, got {n_cov}/{n_tot}")

    lo = 0.0 if n_cov == 0 else float(
        stats.beta.ppf(alpha_ci / 2, n_cov, n_tot - n_cov + 1)
    )
    hi = 1.0 if n_cov == n_tot else float(
        stats.beta.ppf(1 - alpha_ci / 2, n_cov + 1, n_tot - n_cov)
    )
    return lo, hi

print(f"{'Split':<22}  {'Method':<22}  {'MC (%)':>8}  {'CI_lo':>8}  {'CI_hi':>8}  {'Dev (pp)':>9}  Prac.Sig?")
print("=" * 100)

for split in TEST_SPLITS:
    for method in METHODS:
        m = COV_LOGIT[split][method]
        n_covered = round(m["mc"] * m["n"])
        lo, hi    = clopper_pearson(n_covered, m["n"])
        dev_pp    = m["dev_pp"]
        sig       = "YES" if abs(dev_pp) > PRAC_SIG_PP else "no"
        print(f"  {SPLIT_LABELS[split][:20]:<20}  {METH_LABELS[method]:<22}"
              f"  {m['mc']*100:>8.3f}  {lo*100:>8.3f}  {hi*100:>8.3f}"
              f"  {dev_pp:>+9.3f}  {sig}")

Split                   Method                    MC (%)     CI_lo     CI_hi   Dev (pp)  Prac.Sig?
  Subprime (2007–12)    SCP                       91.078    91.068    91.088     +1.078  no
  Subprime (2007–12)    Mondrian (FICO×LTV)       90.327    90.317    90.338     +0.327  no
  Subprime (2007–12)    APS                       89.040    89.029    89.051     -0.960  no
  Normal (2013–Sep 201  SCP                       93.844    93.836    93.851     +3.844  YES
  Normal (2013–Sep 201  Mondrian (FICO×LTV)       94.087    94.079    94.094     +4.087  YES
  Normal (2013–Sep 201  APS                       89.505    89.495    89.515     -0.495  no
  COVID (2020–21)       SCP                       94.293    94.279    94.308     +4.293  YES
  COVID (2020–21)       Mondrian (FICO×LTV)       93.408    93.393    93.423     +3.408  YES
  COVID (2020–21)       APS                       89.185    89.166    89.204     -0.815  no
  Rate Hike (2022–23)   SCP                       95.804    95.794   

---
## Section 5 · Primary Pipeline vs Logit: Robustness Comparison

This section compares the primary LightGBM pipeline with the logistic benchmark. Because scorer, benchmark sample, preprocessing, fitted probability map, and diagnostic sample change jointly, persistence supports robustness to the alternative pipeline. Differences cannot be attributed to model class alone.

In [ ]:
# ── §5.1 · Table A: Discriminative Power ─────────────────────────────────────────

print("=== Table A: Discriminative Power (AUROC, AUPRC, Brier) ===")
print()
print(f"{'Split':<22}  {'Logit AUROC':>12}  {'LGBM AUROC':>12}  {'Δ':>8}  "
      f"{'Logit AUPRC':>12}  {'LGBM AUPRC':>12}  "
      f"{'Logit Brier':>12}  {'LGBM Brier':>12}")
print("-" * 115)

table_a_rows = []
for split in ALL_SPLITS:
    res = EVAL_RESULTS[split]
    la  = res["auroc"];  ga = LGBM_REF["auroc"][split]
    lp  = res["auprc"];  gp = LGBM_REF["auprc"][split]
    lb  = res["brier"];  gb = LGBM_REF["brier"][split]
    print(f"  {SPLIT_LABELS[split][:20]:<20}  {la:>12.4f}  {ga:>12.4f}  {la-ga:>+8.4f}"
          f"  {lp:>12.4f}  {gp:>12.4f}"
          f"  {lb:>12.6f}  {gb:>12.6f}")
    table_a_rows.append({"split":split,"logit_auroc":la,"lgbm_auroc":ga,
                          "logit_auprc":lp,"lgbm_auprc":gp,
                          "logit_brier":lb,"lgbm_brier":gb})

pd.DataFrame(table_a_rows).to_csv(NB10_OUT_DIR / "nb10_table_a_discriminative.csv", index=False)

=== Table A: Discriminative Power (AUROC, AUPRC, Brier) ===

Split                    Logit AUROC    LGBM AUROC         Δ   Logit AUPRC    LGBM AUPRC   Logit Brier    LGBM Brier
-------------------------------------------------------------------------------------------------------------------
  Calibration (2005–06        0.9089        0.9140   -0.0052        0.2865        0.3361      0.066746      0.063955
  Subprime (2007–12)          0.8810        0.8833   -0.0023        0.2961        0.3494      0.072213      0.060493
  Normal (2013–Sep 201        0.8724        0.8723   +0.0000        0.2373        0.3135      0.053625      0.040320
  COVID (2020–21)             0.8216        0.8262   -0.0046        0.1500        0.2114      0.050462      0.038993
  Rate Hike (2022–23)         0.8638        0.8719   -0.0081        0.1403        0.2508      0.040534      0.028879


In [ ]:
# ── §5.2 · Tables B+C: Marginal and Class-Conditional Coverage ──────────────────

print("=== Table B: Marginal Coverage Deviation from 90 % (pp) ===")
print(f"{'Method':<22}  {'Sub-L':>8}  {'Sub-G':>8}  "
      f"{'Norm-L':>8}  {'Norm-G':>8}  "
      f"{'COVID-L':>8}  {'COVID-G':>8}  "
      f"{'RH-L':>8}  {'RH-G':>8}")
print("-" * 90)

lgbm_mc = {"scp": LGBM_REF["mc_scp"], "mondrian_fico_ltv": LGBM_REF["mc_mond"], "aps": LGBM_REF["mc_aps"]}
for method in METHODS:
    row = f"  {METH_LABELS[method]:<20}  "
    for split in TEST_SPLITS:
        logit_dev = COV_LOGIT[split][method]["dev_pp"]
        lgbm_dev  = lgbm_mc[method][split] - 90.0
        row += f"  {logit_dev:>+6.2f}  {lgbm_dev:>+6.2f}"
    print(row)
print("  L = Logit, G = LightGBM")

print()
print("=== Table C: Cov(y=1) % ===")
print(f"{'Method':<22}  {'Sub-L':>10}  {'Sub-G':>10}  "
      f"{'Norm-L':>10}  {'Norm-G':>10}  "
      f"{'COVID-L':>10}  {'COVID-G':>10}  "
      f"{'RH-L':>10}  {'RH-G':>10}")
print("-" * 105)

lgbm_cy1 = {"scp": LGBM_REF["cov_y1_scp"], "aps": LGBM_REF["cov_y1_aps"],
             "mondrian_fico_ltv": LGBM_REF["cov_y1_mond"]}
for method in METHODS:
    row = f"  {METH_LABELS[method]:<20}  "
    for split in TEST_SPLITS:
        logit_cy1 = COV_LOGIT[split][method]["cov_y1"] * 100
        lgbm_cy1_ = lgbm_cy1.get(method, {}).get(split, float("nan"))
        row += f"  {logit_cy1:>10.4f}  {lgbm_cy1_:>10.4f}"
    print(row)

# Save combined table
rows_bc = []
for method in METHODS:
    for split in TEST_SPLITS:
        rows_bc.append({
            "method": method, "split": split,
            "logit_mc_dev_pp": COV_LOGIT[split][method]["dev_pp"],
            "lgbm_mc_dev_pp": lgbm_mc[method][split] - 90.0,
            "logit_cov_y1_pct": COV_LOGIT[split][method]["cov_y1"]*100,
            "lgbm_cov_y1_pct": lgbm_cy1.get(method,{}).get(split, float("nan")),
        })
pd.DataFrame(rows_bc).to_csv(NB10_OUT_DIR / "nb10_table_bc_coverage.csv", index=False)

=== Table B: Marginal Coverage Deviation from 90 % (pp) ===
Method                     Sub-L     Sub-G    Norm-L    Norm-G   COVID-L   COVID-G      RH-L      RH-G
------------------------------------------------------------------------------------------
  SCP                      +1.08   +0.51   +3.84   +4.55   +4.29   +4.83   +5.80   +6.41
  Mondrian (FICO×LTV)      +0.33   -0.50   +4.09   +4.27   +3.41   +3.90   +4.55   +5.29
  APS                      -0.96   -1.01   -0.49   -0.54   -0.81   -0.86   -0.46   -0.42
  L = Logit, G = LightGBM

=== Table C: Cov(y=1) % ===
Method                       Sub-L       Sub-G      Norm-L      Norm-G     COVID-L     COVID-G        RH-L        RH-G
---------------------------------------------------------------------------------------------------------
  SCP                         0.3301      0.0007      0.8036      0.0002      0.1351      0.0000      1.4251      0.0001
  Mondrian (FICO×LTV)         0.6013      0.9752      1.1176      0.7945      

In [ ]:
# ── §5.3 · Tables D+E: CovGap and Abstention Signal ─────────────────────────────

print("=== Table D: CovGap (pp) ===")
print(f"{'Method':<22}  {'Sub-L':>8}  {'Sub-G':>8}  "
      f"{'Norm-L':>8}  {'Norm-G':>8}  "
      f"{'COVID-L':>8}  {'COVID-G':>8}  "
      f"{'RH-L':>8}  {'RH-G':>8}")
print("-" * 90)

lgbm_cg = {"scp": LGBM_REF["covgap_scp"], "mondrian_fico_ltv": LGBM_REF["covgap_mond"],
            "aps": LGBM_REF["covgap_aps"]}
for method in METHODS:
    row = f"  {METH_LABELS[method]:<20}  "
    for split in TEST_SPLITS:
        logit_cg = COV_LOGIT[split][method]["covgap_pp"]
        lgbm_cg_ = lgbm_cg[method][split]
        row += f"  {logit_cg:>8.2f}  {lgbm_cg_:>8.2f}"
    print(row)

print()
print("=== Table E: Empty-Set Rate (Abstention Signal, %) ===")
print(f"{'Method':<22}  {'Sub-L':>8}  {'Sub-G':>8}  "
      f"{'Norm-L':>8}  {'Norm-G':>8}  "
      f"{'COVID-L':>8}  {'COVID-G':>8}  "
      f"{'RH-L':>8}  {'RH-G':>8}")
print("-" * 90)

lgbm_em = {"scp": LGBM_REF["empty_scp"], "mondrian_fico_ltv": LGBM_REF["empty_mond"],
            "aps": LGBM_REF["empty_aps"]}
for method in METHODS:
    row = f"  {METH_LABELS[method]:<20}  "
    for split in TEST_SPLITS:
        logit_em = COV_LOGIT[split][method]["empty_rate"]*100
        lgbm_em_ = lgbm_em[method][split]
        row += f"  {logit_em:>8.2f}  {lgbm_em_:>8.2f}"
    print(row)
print("  L = Logit, G = LightGBM")

rows_de = []
for method in METHODS:
    for split in TEST_SPLITS:
        rows_de.append({
            "method":method,"split":split,
            "logit_covgap_pp": COV_LOGIT[split][method]["covgap_pp"],
            "lgbm_covgap_pp":  lgbm_cg[method][split],
            "logit_empty_pct": COV_LOGIT[split][method]["empty_rate"]*100,
            "lgbm_empty_pct":  lgbm_em[method][split],
        })
pd.DataFrame(rows_de).to_csv(NB10_OUT_DIR / "nb10_table_de_covgap_empty.csv", index=False)

=== Table D: CovGap (pp) ===
Method                     Sub-L     Sub-G    Norm-L    Norm-G   COVID-L   COVID-G      RH-L      RH-G
------------------------------------------------------------------------------------------
  SCP                        76.54     79.69     69.85     47.07     70.04     31.97     67.48     24.94
  Mondrian (FICO×LTV)         7.29      5.26      6.20      7.34      4.79      5.04      5.65      6.15
  APS                         2.96      3.92      2.43      1.28      2.69      1.72      2.76      1.57

=== Table E: Empty-Set Rate (Abstention Signal, %) ===
Method                     Sub-L     Sub-G    Norm-L    Norm-G   COVID-L   COVID-G      RH-L      RH-G
------------------------------------------------------------------------------------------
  SCP                         7.89      8.52      5.48      4.82      4.76      4.29      3.63      3.08
  Mondrian (FICO×LTV)         8.48      9.34      5.16      5.01      5.67      5.18      4.88      4.17
  

In [ ]:
# ── §5.4 · Table F: PSI–Coverage Endpoint Dissociation ────────────────────────
# Tests whether the thesis-level endpoint pattern survives the logistic benchmark:
# Crisis should remain the lowest-PSI / widest-CovGap regime, and Rate Hike the
# highest-PSI / narrowest-CovGap regime. The Normal/COVID middle ordering need
# not match, so this is not a four-regime monotonicity or model-agnosticism test.

print("=== Table F: PSI–Coverage Dissociation (Logit vs LightGBM) ===")
print()
print(f"{'Split':<22}  {'PSI-L':>8}  {'PSI-G':>8}  "
      f"{'SCP MC dev-L':>14}  {'SCP MC dev-G':>14}  {'Mechanism':<20}")
print("-" * 95)

rows_f = []
for split in TEST_SPLITS:
    psi_l   = PSI_LOGIT[split]["psi"]
    psi_g   = LGBM_REF["psi_scp"][split]
    dev_l   = COV_LOGIT[split]["scp"]["dev_pp"]
    dev_g   = LGBM_REF["mc_scp"][split] - 90.0
    mech    = MECH_LABELS[split]
    print(f"  {SPLIT_LABELS[split][:20]:<20}  {psi_l:>8.4f}  {psi_g:>8.4f}"
          f"  {dev_l:>+14.3f} pp  {dev_g:>+14.3f} pp  {mech}")
    rows_f.append({"split":split,"psi_logit":psi_l,"psi_lgbm":psi_g,
                   "scp_mc_dev_logit":dev_l,"scp_mc_dev_lgbm":dev_g,"mechanism":mech})

pd.DataFrame(rows_f).to_csv(NB10_OUT_DIR / "nb10_table_f_psi_dissociation.csv", index=False)

print()
# Check if PSI ordering is preserved (subprime should still have lowest PSI)
psi_vals_logit = {s: PSI_LOGIT[s]["psi"] for s in TEST_SPLITS}
sub_is_lowest  = (psi_vals_logit["test_subprime"] ==
                  min(psi_vals_logit.values()))
print(f"  PSI ordering: subprime is still lowest PSI? {sub_is_lowest}")
print(f"  Logit PSI order: {sorted(psi_vals_logit.items(), key=lambda x: x[1])}")
print()
print(f"  {'Finding':<55}  {'LightGBM':>10}  {'Logit':>10}")
print(f"  {'Subprime PSI (lowest of 4 regimes?)':<55}  "
      f"  {'Yes':>10}  {'Yes' if sub_is_lowest else 'No':>10}")
print(f"  {'Subprime SCP CovGap (worst of 4 regimes?)':<55}  "
      f"  {'Yes':>10}  "
      f"  {'Yes' if COV_LOGIT['test_subprime']['scp']['covgap_pp'] == max(COV_LOGIT[s]['scp']['covgap_pp'] for s in TEST_SPLITS) else 'No':>10}")

=== Table F: PSI–Coverage Dissociation (Logit vs LightGBM) ===

Split                      PSI-L     PSI-G    SCP MC dev-L    SCP MC dev-G  Mechanism           
-----------------------------------------------------------------------------------------------
  Subprime (2007–12)      0.0084    0.0062          +1.078 pp          +0.515 pp  Tail Shift
  Normal (2013–Sep 201    0.1171    0.0891          +3.844 pp          +4.548 pp  Stable
  COVID (2020–21)         0.1669    0.1240          +4.293 pp          +4.827 pp  Structural Shift
  Rate Hike (2022–23)     0.2565    0.1988          +5.804 pp          +6.409 pp  Composition Shift

  PSI ordering: subprime is still lowest PSI? True
  Logit PSI order: [('test_subprime', 0.008407482468239545), ('test_normal', 0.11707636331959405), ('test_covid', 0.16685657079662894), ('test_rate_hike', 0.2564684372607866)]

  Finding                                                    LightGBM       Logit
  Subprime PSI (lowest of 4 regimes?)              

---
## Section 6 · Calibration Contribution: Was Isotonic Recalibration Critical for CP?

This section directly compares **logit-raw-CP** (CP applied to uncalibrated logit probabilities) vs **logit-iso-CP** (CP applied after isotonic post-recalibration). The within-pipeline comparison evaluates the incremental effect of isotonic recalibration on CP coverage metrics.

**Theoretical background:**  
The marginal split-CP guarantee is largely score-agnostic: under exchangeability, it
does not require the base probabilities to be perfectly calibrated. However, probability
calibration still affects the *shape and usefulness* of the prediction sets:
1. **Efficiency / informativeness:** better probability scores can yield smaller or more
   decision-useful prediction sets for the same marginal target.
2. **Class-conditional behaviour:** under severe class imbalance, distorted probability
   scores can make positive-class nonconformity scores systematically large, so the
   global threshold may cover almost no positive cases. Isotonic recalibration can
   change this score geometry, although it does not create a new class-conditional
   validity guarantee by itself.

The raw logit probabilities are themselves strongly miscalibrated, and isotonic recalibration substantially improves their probability calibration. The question here is whether that transformation also materially changes class-conditional coverage and CovGap.


### §6 Calibration contribution to class-conditional coverage

As stated before, isotonic recalibration is **not required for conformal validity**. It is used here to put the benchmark on the probability scale used by APS and by the drift, operational, and cross-model analyses.

Within the logistic benchmark, the raw-versus-isotonic comparison measures the map's incremental effect while holding the benchmark scorer and class ratio fixed. The clearest result is SCP: the map compresses the score support toward the roughly one-percent base rate, moves the global threshold from 0.4347 to 0.0184, and sharply reduces delinquent-class coverage across all four test windows. This is a within-pipeline benchmark effect, not a claim that calibration generally harms conformal prediction.

The SCP contrast is deterministic. The APS raw/isotonic rows use separate deterministic random streams and should therefore be read descriptively rather than as a paired-randomization contrast.

In [ ]:
# ── §6.1 · Coverage comparison: logit-raw-CP vs logit-iso-CP ─────────────────────
# Compute SCP and APS thresholds using RAW logit probabilities (no isotonic).

# SCP scores on cp_cal using raw probabilities
s_scp_raw_cpcal = compute_score_scp(p_cp_cal, y_cp_cal)  # p_cp_cal is raw (from §2.2)
Q_HAT_SCP_RAW   = cp_quantile(s_scp_raw_cpcal, ALPHA)

rng_r  = np.random.default_rng(seed=batch_seed("calibration_raw_cp_cal", 0))
U_r    = rng_r.uniform(size=len(y_cp_cal)).astype(np.float32)
s_aps_raw_cpcal = compute_score_aps(p_cp_cal, y_cp_cal, U_r)
Q_HAT_APS_RAW   = cp_quantile(s_aps_raw_cpcal, ALPHA)

print(f"Threshold comparison (raw vs iso):")
print(f"  q_hat_SCP raw : {Q_HAT_SCP_RAW:.8f}")
print(f"  q_hat_SCP iso : {Q_HAT_SCP_LOGIT:.8f}")
print(f"  q_hat_APS raw : {Q_HAT_APS_RAW:.8f}")
print(f"  q_hat_APS iso : {Q_HAT_APS_LOGIT:.8f}")
print()

# Evaluate on test splits with raw probabilities
RAW_COV: dict = {}
for split in TEST_SPLITS:
    p_raw_t = EVAL_RESULTS[split]["p_hat_raw"]
    y_t     = EVAL_RESULTS[split]["y"]
    n_t     = len(y_t)
    groups  = EVAL_RESULTS[split]["df"]["fico_ltv_group"].to_numpy()

    s_scp_r = compute_score_scp(p_raw_t, y_t)
    U_ts = np.empty(n_t, dtype=np.float32)
    obs_years_raw = EVAL_RESULTS[split]["df"]["obs_year"].to_numpy()
    for yr in np.unique(obs_years_raw):
        m_yr = obs_years_raw == yr
        rng_yr = np.random.default_rng(
            seed=batch_seed(f"{split}_raw", int(yr)))
        U_ts[m_yr] = rng_yr.uniform(size=m_yr.sum()).astype(np.float32)
    s_aps_r = compute_score_aps(p_raw_t, y_t, U_ts)

    pos_mask = y_t == 1
    RAW_COV[split] = {}
    for method, scores, qhat in [
        ("scp", s_scp_r, Q_HAT_SCP_RAW),
        ("aps", s_aps_r, Q_HAT_APS_RAW),
    ]:
        covered = scores <= np.float32(qhat)
        cell_covs = {}
        cell_n_raw = {}
        for c in np.unique(groups):
            mask_c = groups == c
            n_c = int(mask_c.sum())
            if n_c > 0:
                cell_covs[c] = float(covered[mask_c].mean())
                cell_n_raw[c] = n_c
        valid_c = [v for k, v in cell_covs.items()
                   if cell_n_raw[k] >= COVGAP_MIN_CELL_N]
        # Empty-set rate depends on method
        if method == "scp":
            empty_m = float(((p_raw_t > float(np.float32(qhat))) &
                             (p_raw_t < (1.0 - float(np.float32(qhat))))).mean())
        else:  # aps
            U_alt_raw = np.empty(n_t, dtype=np.float32)
            for yr in np.unique(obs_years_raw):
                m_yr = obs_years_raw == yr
                rng_alt_yr = np.random.default_rng(
                    seed=batch_seed(f"{split}_raw", int(yr)) ^ 0x80000000)
                U_alt_raw[m_yr] = rng_alt_yr.uniform(size=m_yr.sum()).astype(np.float32)
            y_alt_raw = (1 - y_t).astype(np.int32)
            s_alt_raw = compute_score_aps(p_raw_t, y_alt_raw, U_alt_raw)
            empty_m = float(((s_aps_r > float(np.float32(qhat))) &
                             (s_alt_raw > float(np.float32(qhat)))).mean())

        RAW_COV[split][method] = {
            "mc": float(covered.mean()),
            "cov_y1": float(covered[pos_mask].mean()),
            "empty_rate": empty_m,
            "covgap_pp": (max(abs(v - NOMINAL_COV) * 100 for v in valid_c)
              if valid_c else float("nan")),
        }

print(f"{'':60}  {'SCP':>25}  {'APS':>25}")
print(f"{'Split':<22}  {'Metric':<18}  "
      f"{'Raw':>8}  {'Iso':>8}  {'Δ':>8}  "
      f"{'Raw':>8}  {'Iso':>8}  {'Δ':>8}")
print("-" * 110)
for split in TEST_SPLITS:
    for metric, key in [("MC %", "mc"), ("Cov(y=1)%","cov_y1"), ("CovGap pp","covgap_pp")]:
        r_scp = RAW_COV[split].get("scp",{}).get(key, float("nan"))
        i_scp = COV_LOGIT[split]["scp"][key]
        r_aps = RAW_COV[split].get("aps",{}).get(key, float("nan"))
        i_aps = COV_LOGIT[split]["aps"][key]
        scale = 100 if key in ("mc","cov_y1","empty_rate") else 1
        print(f"  {SPLIT_LABELS[split][:18]:<18}  {metric:<18}  "
              f"  {r_scp*scale:>8.3f}  {i_scp*scale:>8.3f}  {(i_scp-r_scp)*scale:>+8.3f}"
              f"  {r_aps*scale:>8.3f}  {i_aps*scale:>8.3f}  {(i_aps-r_aps)*scale:>+8.3f}")
    print()

Threshold comparison (raw vs iso):
  q_hat_SCP raw : 0.43470857
  q_hat_SCP iso : 0.01836919
  q_hat_APS raw : 0.85041320
  q_hat_APS iso : 0.89983147

                                                                                    SCP                        APS
Split                   Metric                   Raw       Iso         Δ       Raw       Iso         Δ
--------------------------------------------------------------------------------------------------------------
  Subprime (2007–12)  MC %                    89.116    91.078    +1.962    88.853    89.040    +0.187
  Subprime (2007–12)  Cov(y=1)%               58.598     0.330   -58.268    69.265    27.734   -41.531
  Subprime (2007–12)  CovGap pp               70.873    76.542    +5.669    41.795     2.957   -38.838

  Normal (2013–Sep 2  MC %                    92.508    93.844    +1.336    88.766    89.505    +0.739
  Normal (2013–Sep 2  Cov(y=1)%               57.031     0.804   -56.227    64.025    28.969   -35.056
  N

In [ ]:
# ── §6.2 · Score distribution data: raw vs iso (logit and LightGBM reference) ────

p_cal_logit_raw = EVAL_RESULTS["calibration"]["p_hat_raw"]
p_cal_logit_iso = EVAL_RESULTS["calibration"]["p_hat_iso"]

bins = np.linspace(0, 1, 51)  # 50 bins of width 0.02
hist_raw, _ = np.histogram(p_cal_logit_raw, bins=bins, density=True)
hist_iso, _ = np.histogram(p_cal_logit_iso, bins=bins, density=True)
bin_centres  = (bins[:-1] + bins[1:]) / 2

print("Logit probability distribution statistics (calibration, 5 % sample):")
print(f"  RAW:  mean={p_cal_logit_raw.mean():.6f}  std={p_cal_logit_raw.std():.6f}"
      f"  p<0.01: {(p_cal_logit_raw<0.01).mean()*100:.1f}%"
      f"  p>0.50: {(p_cal_logit_raw>0.50).mean()*100:.2f}%")
print(f"  ISO:  mean={p_cal_logit_iso.mean():.6f}  std={p_cal_logit_iso.std():.6f}"
      f"  p<0.01: {(p_cal_logit_iso<0.01).mean()*100:.1f}%"
      f"  p>0.50: {(p_cal_logit_iso>0.50).mean()*100:.2f}%")
print()

if LGBM_PAVA_BP is not None:
    print(f"  ISO:  smooth monotone correction  ({LGBM_PAVA_BP} PAVA breakpoints)")
else:
    print("  ISO:  smooth monotone correction")

# Save histogram data
hist_df = pd.DataFrame({
    "bin_centre": bin_centres,
    "density_logit_raw": hist_raw,
    "density_logit_iso": hist_iso,
})
hist_df.to_csv(NB10_OUT_DIR / "nb10_score_distribution.csv", index=False)

Logit probability distribution statistics (calibration, 5 % sample):
  RAW:  mean=0.177285  std=0.199675  p<0.01: 1.6%  p>0.50: 7.99%
  ISO:  mean=0.011147  std=0.044123  p<0.01: 82.2%  p>0.50: 0.20%

  ISO:  smooth monotone correction  (516 PAVA breakpoints)


---
## Section 7 · Operational Translation

This section re-evaluates the NB09 decision-layer findings under the logistic benchmark.

1. **SCP and deterministic APS:** at each method's natural **non-auto-clear budget**, routing is a one-sided threshold in `p̂_cal`. Matching the same row budget therefore selects the same upper-tail rows as the probability-ranking baseline, so `Δ = 0` is structural rather than an empirical gain.
2. **Mondrian:** cell-specific thresholds break the global probability ordering, so the informative result is its negative matched-budget capture gap. The detailed FICO×LTV allocation mechanism is established in NB09 and is not re-estimated here.
3. **APS routing:** operational APS uses the deterministic `U = 1` reduction. These routing metrics are therefore not directly comparable with randomized APS coverage and set-composition metrics from §4.

**Legacy-output terminology.** In the stored §7 output, `RV%` and “Equal-Review-Rate” denote the **non-auto-clear share** (`review ∪ high_risk`), and `RvPrec%` is the delinquency rate within that non-auto-cleared set.

In [ ]:
# ── §7.1 · Routing metrics ──────────────────────────────────────────────────────

Q_SCP_F32  = np.float32(Q_HAT_SCP_LOGIT)
Q_APS_F32  = np.float32(Q_HAT_APS_LOGIT)
APS_AUTO_THRESH = np.float32(1.0 - float(Q_APS_F32))   # auto_clear if p < 1 - q_APS
APS_HR_THRESH   = Q_APS_F32                            # high_risk if p > q_APS

print(f"Routing thresholds (logit):")
print(f"  SCP auto_clear : p ≤ {float(Q_SCP_F32):.6f}")
print(f"  SCP high_risk  : p ≥ {1.0 - float(Q_SCP_F32):.6f}")
print(f"  APS auto_clear : p < {float(APS_AUTO_THRESH):.6f}  "
      f"({float(APS_AUTO_THRESH)/float(Q_SCP_F32):.2f}× SCP threshold)")
print(f"  APS high_risk  : p > {float(APS_HR_THRESH):.6f}")
print()

ROUTE_RESULTS: dict = {}

for split in TEST_SPLITS:
    p_iso_t = EVAL_RESULTS[split]["p_hat_iso"]
    y_t     = EVAL_RESULTS[split]["y"]
    groups  = EVAL_RESULTS[split]["df"]["fico_ltv_group"].to_numpy()
    n_t     = len(y_t)
    pos_rt  = y_t.mean()

    ROUTE_RESULTS[split] = {}
    for method in METHODS:
        if method == "scp":
            q = float(Q_SCP_F32)
            hi_thresh = 1.0 - q
            auto  = p_iso_t <= q
            high  = p_iso_t >= hi_thresh
            rev   = ~auto & ~high
        elif method == "mondrian_fico_ltv":
            q_per = np.array([MONDRIAN_LOGIT.get(g, {"q_hat": float(Q_SCP_F32)})["q_hat"]
                              for g in groups], dtype=np.float32)
            auto  = p_iso_t <= q_per
            high  = p_iso_t >= (1.0 - q_per)
            rev   = ~auto & ~high
        else:  # aps
            auto  = p_iso_t < APS_AUTO_THRESH
            high  = p_iso_t > APS_HR_THRESH
            rev   = ~auto & ~high

        fac     = float((auto & (y_t==1)).sum()) / max(1, int((y_t==1).sum()))
        capture = float(((rev | high) & (y_t==1)).sum()) / max(1, int((y_t==1).sum()))
        rv_prec = float((y_t[rev | high]).mean()) if (rev | high).sum() > 0 else float("nan")
        rv_lift = rv_prec / pos_rt if pos_rt > 0 else float("nan")
        rv_rate = float((rev | high).mean())

        ROUTE_RESULTS[split][method] = {
            "ac_rate": float(auto.mean()), "rv_rate": rv_rate,
            "fac": fac, "capture": capture,
            "rv_precision": rv_prec, "rv_lift": rv_lift,
        }

print(f"{'Method':<22}  {'Split':<22}  {'AC%':>6}  {'RV%':>6}  "
      f"{'FAC%':>8}  {'Capt%':>8}  {'RvPrec%':>8}  {'RvLift':>8}")
print("-" * 100)
for method in METHODS:
    for split in TEST_SPLITS:
        r = ROUTE_RESULTS[split][method]
        print(f"  {METH_LABELS[method]:<20}  {SPLIT_LABELS[split][:20]:<20}"
              f"  {r['ac_rate']*100:>6.2f}  {r['rv_rate']*100:>6.2f}"
              f"  {r['fac']*100:>8.3f}  {r['capture']*100:>8.3f}"
              f"  {r['rv_precision']*100:>8.3f}  {r['rv_lift']:>8.2f}x")
    print()

Routing thresholds (logit):
  SCP auto_clear : p ≤ 0.018369
  SCP high_risk  : p ≥ 0.981631
  APS auto_clear : p < 0.100169  (5.45× SCP threshold)
  APS high_risk  : p > 0.899831

Method                  Split                      AC%     RV%      FAC%     Capt%   RvPrec%    RvLift
----------------------------------------------------------------------------------------------------
  SCP                   Subprime (2007–12)     92.09    7.91    40.118    59.882    19.155      7.57x
  SCP                   Normal (2013–Sep 201   94.47    5.53    42.025    57.975    16.023     10.49x
  SCP                   COVID (2020–21)        95.23    4.77    56.351    43.649    15.257      9.15x
  SCP                   Rate Hike (2022–23)    96.31    3.69    50.487    49.513    13.976     13.44x

  Mondrian (FICO×LTV)   Subprime (2007–12)     91.49    8.51    46.495    53.505    15.916      6.29x
  Mondrian (FICO×LTV)   Normal (2013–Sep 201   94.79    5.21    46.858    53.142    15.565     10.19x
  M

In [ ]:
# ── §7.2 · Equal-review-rate Δ test ─────────────────────────────────────────────
# For each (method, regime): at the method's natural review rate, how much
# delinquency does CP capture compared to ranking loans by descending p̂ at
# the same rate?

delta_rows = []
for method in METHODS:
    for split in TEST_SPLITS:
        r       = ROUTE_RESULTS[split][method]
        y_t     = EVAL_RESULTS[split]["y"]
        p_iso_t = EVAL_RESULTS[split]["p_hat_iso"]
        rv_rate = r["rv_rate"]

        # Probability-threshold baseline: take top rv_rate fraction by p̂
        n_review_base = max(1, round(rv_rate * len(y_t)))
        rank_idx      = np.argsort(p_iso_t)[::-1][:n_review_base]
        baseline_cap  = float(y_t[rank_idx].sum()) / max(1, int(y_t.sum()))

        cp_cap  = r["capture"]
        delta   = (cp_cap - baseline_cap) * 100   # pp

        lgbm_delta = NB09_DELTA_REF.get(method, {}).get(split, float("nan"))

        delta_rows.append({
            "method": method, "split": split,
            "rv_rate_pct": rv_rate*100,
            "cp_capture_pct": cp_cap*100,
            "baseline_capture_pct": baseline_cap*100,
            "delta_pp_logit": delta,
            "delta_pp_lgbm": lgbm_delta,
        })

df_delta = pd.DataFrame(delta_rows)

print("=== Equal-Review-Rate Comparison: CP Capture - Baseline Threshold Capture (pp) ===")
print()
print(f"{'Method':<22}  {'Split':<22}  {'RV%':>6}  "
      f"{'CP Cap%':>8}  {'BL Cap%':>8}  {'Δ-Logit':>9}  {'Δ-LGBM':>9}")
print("-" * 100)
for _, row in df_delta.iterrows():
    print(f"  {METH_LABELS[row['method']]:<20}  {SPLIT_LABELS[row['split']][:20]:<20}"
          f"  {row['rv_rate_pct']:>6.2f}  {row['cp_capture_pct']:>8.3f}"
          f"  {row['baseline_capture_pct']:>8.3f}  {row['delta_pp_logit']:>+9.3f}"
          f"  {row['delta_pp_lgbm']:>+9.3f}")

df_delta.to_csv(NB10_OUT_DIR / "nb10_operational_delta.csv", index=False)

print()
print("Summary of Δ findings:")
for method in METHODS:
    deltas_l = [r["delta_pp_logit"] for _, r in df_delta[df_delta["method"]==method].iterrows()]
    deltas_g = [r["delta_pp_lgbm"]  for _, r in df_delta[df_delta["method"]==method].iterrows()]
    print(f"  {METH_LABELS[method]:<22}  Logit mean Δ = {np.mean(deltas_l):+.3f} pp  |  "
          f"LGBM mean Δ = {np.nanmean(deltas_g):+.3f} pp")

=== Equal-Review-Rate Comparison: CP Capture - Baseline Threshold Capture (pp) ===

Method                  Split                      RV%   CP Cap%   BL Cap%    Δ-Logit     Δ-LGBM
----------------------------------------------------------------------------------------------------
  SCP                   Subprime (2007–12)      7.91    59.882    59.882     +0.000     +0.000
  SCP                   Normal (2013–Sep 201    5.53    57.975    57.975     +0.000     +0.000
  SCP                   COVID (2020–21)         4.77    43.649    43.649     +0.000     +0.000
  SCP                   Rate Hike (2022–23)     3.69    49.513    49.513     +0.000     +0.000
  Mondrian (FICO×LTV)   Subprime (2007–12)      8.51    53.505    60.871     -7.365     -8.739
  Mondrian (FICO×LTV)   Normal (2013–Sep 201    5.21    53.142    57.165     -4.024     -5.818
  Mondrian (FICO×LTV)   COVID (2020–21)         5.68    45.085    46.847     -1.762     -3.732
  Mondrian (FICO×LTV)   Rate Hike (2022–23)     4.93 

---
## Section 8 · Findings


In [ ]:
# ── §8.1 · Hypothesis assessment ────────────────────────────────────────────────

print("=" * 80)
print("NB10 HYPOTHESIS ASSESSMENT")
print("=" * 80)

# H10.1 -- APS near-nominal marginal coverage
aps_devs = [abs(COV_LOGIT[s]["aps"]["dev_pp"]) for s in TEST_SPLITS]
h1_status = "CONFIRMED" if max(aps_devs) <= PRAC_SIG_PP else "PARTIAL"
print(f"\nH10.1 -- APS near-nominal marginal coverage with logit: {h1_status}")
for s in TEST_SPLITS:
    print(f"    {s}: {COV_LOGIT[s]['aps']['dev_pp']:+.3f} pp  (|{abs(COV_LOGIT[s]['aps']['dev_pp']):.3f}| ≤ {PRAC_SIG_PP} pp ?  {'Yes' if abs(COV_LOGIT[s]['aps']['dev_pp']) <= PRAC_SIG_PP else 'No'})")

# H10.2 -- SCP class-conditional collapse
max_cov_y1_scp = max(COV_LOGIT[s]["scp"]["cov_y1"]*100 for s in TEST_SPLITS)
h2_status = "CONFIRMED" if max_cov_y1_scp < 5.0 else "REJECTED"
print(f"\nH10.2 -- SCP Cov(y=1) ≈ 0 with logit: {h2_status}")
for s in TEST_SPLITS:
    print(f"    {s}: Cov(y=1) = {COV_LOGIT[s]['scp']['cov_y1']*100:.4f}%")

# H10.3 -- Mondrian threshold spread
vals_m = [v["q_hat"] for v in MONDRIAN_LOGIT.values() if v["n_rows"] >= MONDRIAN_MIN_ROWS]
spread_logit = max(vals_m) / min(vals_m) if min(vals_m) > 0 else float("nan")
h3_status = "CONFIRMED" if spread_logit > 10.0 else "REJECTED"
print(f"\nH10.3 -- Mondrian FICO×LTV spread remains large: {h3_status}")
print(f"    Logit spread: {spread_logit:.2f}×  (LightGBM: {LGBM_MONDRIAN_SPREAD:.2f}×)")

# H10.4 -- PSI dissociation
psi_order_logit = sorted(PSI_LOGIT.keys(), key=lambda s: PSI_LOGIT[s]["psi"])
scp_cg_order    = sorted(TEST_SPLITS, key=lambda s: -COV_LOGIT[s]["scp"]["covgap_pp"])
h4_status = "CONFIRMED" if psi_order_logit[0] == "test_subprime" else "REJECTED"
print(f"\nH10.4 -- PSI–Coverage dissociation replicates: {h4_status}")
print(f"    PSI ascending order (logit): {psi_order_logit}")
print(f"    CovGap descending order    : {scp_cg_order}")

# H10.5 -- Delta ≈ 0 for SCP and APS
max_delta_scp = float(df_delta[df_delta["method"] == "scp"]["delta_pp_logit"].abs().max())
max_delta_aps = float(df_delta[df_delta["method"] == "aps"]["delta_pp_logit"].abs().max())
h5_status = "CONFIRMED" if max_delta_scp < 1.0 and max_delta_aps < 1.0 else "PARTIAL"
print(f"\nH10.5 -- Δ ≈ 0 for SCP and APS at equal budget: {h5_status}")
print(f"    Max |Δ| SCP (logit): {max_delta_scp:.3f} pp  (LightGBM: ≈ 0 pp)")
print(f"    Max |Δ| APS (logit): {max_delta_aps:.3f} pp  (LightGBM: ≈ 0 pp)")
deltas_mond = df_delta[df_delta["method"]=="mondrian_fico_ltv"]["delta_pp_logit"].tolist()
print(f"    Mondrian Δ range   : {min(deltas_mond):.3f} to {max(deltas_mond):.3f} pp  "
          f"(LightGBM: {min(NB09_DELTA_REF['mondrian_fico_ltv'].values()):.1f} to "
          f"{max(NB09_DELTA_REF['mondrian_fico_ltv'].values()):.1f} pp)")

# H10.6 -- Isotonic calibration impact (MARGINAL and CLASS-CONDITIONAL - §6.1 shows the
# largest effect is on Cov(y=1), which marginal coverage alone hides).
scp_mc_raw_devs   = [abs(RAW_COV[s]["scp"]["mc"] - NOMINAL_COV)*100 for s in TEST_SPLITS]
scp_mc_iso_devs   = [abs(COV_LOGIT[s]["scp"]["dev_pp"]) for s in TEST_SPLITS]
avg_iso_mc_impact = np.mean([abs(a-b) for a,b in zip(scp_mc_raw_devs, scp_mc_iso_devs)])
scp_cy1_impact_pp = [abs(COV_LOGIT[s]["scp"]["cov_y1"] - RAW_COV[s]["scp"]["cov_y1"])*100
                     for s in TEST_SPLITS]
max_cy1_impact = max(scp_cy1_impact_pp)
mc_small  = avg_iso_mc_impact < 1.0
cy1_small = max_cy1_impact < PRAC_SIG_PP   # 2 pp
if mc_small and cy1_small:
    h6_status = "CONFIRMED"
elif mc_small and not cy1_small:
    h6_status = "REJECTED (marginal impact small, but LARGE class-conditional impact)"
else:
    h6_status = "REJECTED"
print(f"\nH10.6 -- Isotonic recalibration has small impact on logit CP: {h6_status}")
print(f"    Avg |MC dev change| raw → iso (SCP)   : {avg_iso_mc_impact:.3f} pp")
print(f"    Max |Cov(y=1) change| raw → iso (SCP) : {max_cy1_impact:.3f} pp")
for s in TEST_SPLITS:
    print(f"      {s}: SCP Cov(y=1) {RAW_COV[s]['scp']['cov_y1']*100:.4f}% (raw-CP) "
          f"→ {COV_LOGIT[s]['scp']['cov_y1']*100:.4f}% (iso-CP)  "
          f"[Δ {(COV_LOGIT[s]['scp']['cov_y1']-RAW_COV[s]['scp']['cov_y1'])*100:+.2f} pp]")

NB10 HYPOTHESIS ASSESSMENT

H10.1 -- APS near-nominal marginal coverage with logit: CONFIRMED
    test_subprime: -0.960 pp  (|0.960| ≤ 2.0 pp ?  Yes)
    test_normal: -0.495 pp  (|0.495| ≤ 2.0 pp ?  Yes)
    test_covid: -0.815 pp  (|0.815| ≤ 2.0 pp ?  Yes)
    test_rate_hike: -0.465 pp  (|0.465| ≤ 2.0 pp ?  Yes)

H10.2 -- SCP Cov(y=1) ≈ 0 with logit: CONFIRMED
    test_subprime: Cov(y=1) = 0.3301%
    test_normal: Cov(y=1) = 0.8036%
    test_covid: Cov(y=1) = 0.1351%
    test_rate_hike: Cov(y=1) = 1.4251%

H10.3 -- Mondrian FICO×LTV spread remains large: CONFIRMED
    Logit spread: 83.10×  (LightGBM: 128.71×)

H10.4 -- PSI–Coverage dissociation replicates: CONFIRMED
    PSI ascending order (logit): ['test_subprime', 'test_normal', 'test_covid', 'test_rate_hike']
    CovGap descending order    : ['test_subprime', 'test_covid', 'test_normal', 'test_rate_hike']

H10.5 -- Δ ≈ 0 for SCP and APS at equal budget: CONFIRMED
    Max |Δ| SCP (logit): 0.000 pp  (LightGBM: ≈ 0 pp)
    Max |Δ| APS 

### Locked-findings interpretation note

- **H10.4:** read the stored `CONFIRMED` label as **endpoint replication only**. Crisis is the lowest-PSI / widest-CovGap endpoint and Rate Hike the highest-PSI / narrowest-CovGap endpoint; Normal and COVID reverse their middle ordering. This does not establish a monotone four-regime PSI–coverage relation or model-agnosticism.

- **H10.6:** the stored `0.943 pp` is the mean change in **absolute deviation from 90%**, not the mean raw→isotonic change in marginal coverage. From §6.1, the SCP marginal changes are `+1.962`, `+1.336`, `+1.114`, and `+1.128` pp, with mean absolute change about **1.385 pp**. The overall H10.6 conclusion remains **REJECTED** because the class-conditional effect is large; only the phrase “marginal impact small” is incorrect under the notebook's `<1 pp` rule.

- **H10.5:** “equal budget” means matched **non-auto-clear** budget.

## Appendix · References

### Conformal Prediction - Foundations

- Papadopoulos, H., Proedrou, K., Vovk, V., & Gammerman, A. (2002). Inductive confidence machines for regression. In T. Elomaa, H. Mannila, & H. Toivonen (Eds.), Machine learning: Ecml 2002 (pp. 345–356, Vol. 2430). Springer. https://doi.org/10.1007/3-540-36755-1_29

- Vovk, V., Gammerman, A., & Shafer, G. (2005). Algorithmic learning in a random world (1st ed.). Springer. https://doi.org/10.1007/b106715

- Lei, J., & Wasserman, L. (2014). Distribution-free prediction bands for non-parametric regression. Journal of the Royal Statistical Society Series B: Statistical Methodology, 76(1), 71–96. https://doi.org/10.1111/rssb.12021

### Conformal Prediction - Group-Conditional (Mondrian)

- Vovk, V. (2012). Conditional validity of inductive conformal predictors. In S. C. H. Hoi & W. Buntine (Eds.), Proceedings of the asian conference on machine learning (pp. 475–490, Vol. 25). PMLR. https://proceedings.mlr.press/v25/vovk12.html

### Conformal Prediction - Adaptive Prediction Sets (APS / GIQ)

- Romano, Y., Sesia, M., & Candès, E. J. (2020). Classification with valid and adaptive coverage. In H. Larochelle, M. Ranzato, R. Hadsell, M.-F. Balcan, & H.-T. Lin (Eds.), Advances in neural information processing systems (pp. 3581–3591, Vol. 33). Curran Associates, Inc. https://proceedings.neurips.cc/paper/2020/hash/244edd7e85dc81602b7615cd705545f5-Abstract.html

- Sadinle, M., Lei, J., & Wasserman, L. (2019). Least ambiguous set-valued classifiers with bounded error levels. Journal of the American Statistical Association, 114(525), 223–234. https://doi.org/10.1080/01621459.2017.1395341

### Probability Calibration and Isotonic Regression

- Brier, G. W. (1950). Verification of forecasts expressed in terms of probability. Monthly Weather Review, 78(1), 1–3. https://journals.ametsoc.org/view/journals/mwre/78/1/1520-0493_1950_078_0001_vofeit_2_0_co_2.xml

- Guo, C., Pleiss, G., Sun, Y., & Weinberger, K. Q. (2017). On calibration of modern neural networks. In D. Precup & Y. W. Teh (Eds.), Proceedings of the 34th international conference on machine learning (pp. 1321–1330, Vol. 70). PMLR. https://proceedings.mlr.press/v70/guo17a.html

- Niculescu-Mizil, A., & Caruana, R. (2005). Predicting good probabilities with supervised learning. Proceedings of the 22nd International Conference on Machine Learning, 625–632. https://doi.org/10.1145/1102351.1102430

- Ayer, M., Brunk, H. D., Ewing, G. M., Reid, W. T., & Silverman, E. (1955). An empirical distribution
function for sampling with incomplete information. The Annals of Mathematical Statistics, 26(4), 641–647. https://doi.org/10.1214/aoms/1177728423

### Discrete-Time Hazard / Pooled Logit Benchmark

- Shumway, T. (2001). Forecasting bankruptcy more accurately: A simple hazard model. The Journal
of Business, 74(1), 101–124. https://doi.org/10.1086/209665

- Singer, J. D., & Willett, J. B. (1993). It’s about time: Using discrete-time survival analysis to study
duration and the timing of events. Journal of Educational Statistics, 18(2), 155–195. https://doi.org/10.3102/10769986018002155

### Credit-Risk Benchmarking

- Fitzpatrick, T., & Mues, C. (2016). An empirical comparison of classification algorithms for mort-
gage default prediction: Evidence from a distressed mortgage market. European Journal of
Operational Research, 249(2), 427–439. https://doi.org/10.1016/j.ejor.2015.09.014

- Lessmann, S., Baesens, B., Seow, H.-V., & Thomas, L. C. (2015). Benchmarking state-of-the-art classi-
fication algorithms for credit scoring: An update of research. European Journal of Operational
Research, 247(1), 124–136. https://doi.org/10.1016/j.ejor.2015.05.030

### Regularisation and Linear Models

- Hastie, T., Tibshirani, R., & Friedman, J. (2009). The elements of statistical learning: Data mining,
inference, and prediction (2nd ed.). Springer. https://doi.org/10.1007/978-0-387-84858-7

- Liu, D. C., & Nocedal, J. (1989). On the limited memory BFGS method for large scale optimization.
Mathematical Programming, 45, 503–528. https://doi.org/10.1007/BF01589116

### Statistical Inference and Confidence Intervals

- Clopper, C. J., & Pearson, E. S. (1934). The use of confidence or fiducial limits illustrated in the case
of the binomial. Biometrika, 26(4), 404–413. https://doi.org/10.1093/biomet/26.4.404

### Score Distribution Monitoring (PSI)

- Yurdakul, B., & Naranjo, J. D. (2020). Statistical properties of the population stability index. Journal
of Risk Model Validation, 14(4), 89–100. https://doi.org/10.21314/JRMV.2020.227

### ML vs Econometric Prediction Framing

- Mullainathan, S., & Spiess, J. (2017). Machine learning: An applied econometric approach. Journal of
Economic Perspectives, 31(2), 87–106. https://doi.org/10.1257/jep.31.2.87

### Operational Decision Layer / Reject-Option

- Chow, C. K. (1957). An optimum character recognition system using decision functions. IRE Trans-
actions on Electronic Computers, EC-6(4), 247–254. https://doi.org/10.1109/TEC.1957.5222035

- Chow, C. K. (1970). On optimum recognition error and reject tradeoff. IEEE Transactions on Informa-
tion Theory, 16(1), 41–46. https://doi.org/10.1109/TIT.1970.1054406